# CopyLane 판정 인코더 파인튜닝 v9 — dev 실제 문구 · 문턱 규칙 robust · test 1회 (박수진, 2026-10-06)

> 골든 `529c970556d7` · 8,383행(학습 5,894 · 평가 2,489) · 사전 506종 · D-296 개정 2 · D-321(확정 8종 · 9칸)
> 학습 · 채점 코드는 v8.3 그대로다. **v8 최종 test(후보 A)에서 바꾼 것은 둘**이다 — dev 정의와 문턱 규칙.

## v8 최종(후보 A) → v9 에서 바꾼 것

| # | 바꾼 것 | 후보 A (v8.2) | v9 | 근거 |
|:-:|---|---|---|---|
| 1 | **dev 정의** | 합성(주입) 포함 463행 · 6종 층화 | **실제 문구만 341행 · 8종 층화** (v8.3 R0 과 같다) | dev 의 30% 가 합성 행이었고 모델이 전부 맞혔다 — 문턱이 합성 문구에 맞춰졌다 (test 전 dev 실험에서 확인 · dev 실험 정리 4절) |
| 2 | **문턱 규칙** | 고른 시드 하나의 dev F1 최대 (`best_seed`) | **시드 3개의 dev F1 평균이 최대인 값** (`robust`) | dev 유형별 양성이 33 ~ 52건이라 시드 하나로 고르면 문턱이 흔들린다 — 같은 모델을 다시 학습해도 거짓_과장 문턱이 0.425 → 0.375 로 달라졌다 (dev 실험 정리 8절 5번 · 「다음 차수에서 다룰 것」) |

- 학습 행 · 손실 · 학습률 · epoch · 시드 · 모델 선택 기준(기존 6종 dev macro PR-AUC)은 그대로다.
- 🔴 **문턱 값은 여전히 dev 에서만 나온다.** test 를 보고 고른 값은 없다. 다만 **이 두 가지를 바꾸기로 한 결정은 후보 A 의 test 결과를 본 뒤에 했다** — 보고서에 그렇게 적는다.
- 🔴 **이 골든 판 test 의 5번째 채점이다** (소성민 v10 KcBERT · v10 KcELECTRA · v11 · 박수진 후보 A · v9). 탐색 노트북에서 R0 을 채점했다면 6번째다.
- `THRESHOLD_MODE` 를 `"best_seed"` 로 되돌리면 v8.3 R0 과 같은 판이 된다. **바꾸려면 test 전에** 바꾼다.

## 무엇이 그대로여야 하나 — R0 재현

v9 는 R0(v8.3 기준 실행)과 **같은 학습**이다. 다시 학습하므로 GPU 연산 순서 때문에 소수 셋째 자리가 달라질 수 있지만, 6종 dev macro PR-AUC 가 R0 의 **0.794 ± 0.006** 근처로 나와야 한다. 섹션 9 가 「R0 재현」을 찍는다.

- R0 의 dev: 341행 · 지문 `21dfd0c99500f8ec` · 6종 양성 249 · 음성 76 · 위반 재현율 235/249 · 음성 오판정 17/76 (best_seed 문턱)
- R0 의 문턱(시드 43): 질병 0.525 · 건기식 0.400 · 의약품 0.375 · 거짓_과장 0.500 · 소비자_기만 0.450 · 후기 0.375 · 부당 비교 0.500(고정) · 비방 0.700

## 실행 순서

| 회차 | 설정 (섹션 3) | 볼 것 |
|:-:|---|---|
| **A** | **`FINAL_TEST = False`** ← 🟢 지금 설정 | 섹션 6 — 문턱 두 규칙의 값과 **dev 에서의 차이**(재현율 · 음성 오판정) · 섹션 9 — R0 재현 · 사전 기준 |
| **B** | `FINAL_TEST = True` 만 바꾸고 **런타임을 끊지 않고** 섹션 3부터 다시 실행 | 🔴 test 채점. 다시 학습하지 않는다(`♻️`) |

- 업로드(섹션 1): `golden.jsonl` · `banned_terms.jsonl`.
- 저장 이름: `copylane-encoder-kcbert-v9-all-낱말제외-확정8-dev실제-dev새2종-문턱robust.zip` — v8 의 zip 을 덮지 않는다.
- B 회차가 끝나면 Drive 에 `…_행별확률.csv` · `…_dev행별확률.csv` 가 함께 남는다(문구 없음).

## 사전 고정 기준 (돌리기 전에 적는다)

| # | 조건 | 어디서 |
|---|---|---|
| 재현 | 6종 dev macro PR-AUC 가 R0 과 0.003 이내 | 섹션 9 |
| G1 ~ G3 | R0(best_seed 문턱) 대비 dev 위반 재현율 하락 2%p · 유형별 5%p · 음성 오판정 +1줄 — **문턱 규칙을 바꾼 효과로 읽는다**. 미달이면 미달로 적는다 | 섹션 9 |
| 문턱 | `robust` — 시드 3개의 dev F1 평균 최대 · 격자 0.1 ~ 0.9(0.025) · dev 양성 10건 미만이면 0.5 · 편입 대기 1.01 | 섹션 6 |
| test | 정답 30건 이상인 유형마다 모델 재현율 ≥ 기준선 B(사전 단독판정) + 0.15 · 확정 8종. 미달은 「미달」로 보고한다 | 섹션 8 |

## ⚠️ 알고 볼 것

- **문턱 규칙을 바꾼다고 모델의 실력이 바뀌지 않는다.** 후보 A · 소성민 v10 · v11 의 test macro PR-AUC 가 0.525 · 0.540 · 0.530 으로 같은 수준이었다. v9 에서 달라지는 것은 **얼마나 울리느냐**(재현율과 오탐이 함께 움직인다)다.
- robust 문턱이 best_seed 보다 높을지 낮을지는 돌려 봐야 안다. A 회차의 섹션 6 표가 그 차이를 dev 로 보여 준다.
- dev 는 학습과 같은 원천이다 — 후보 A 에서 유형별 재현율이 test 에서 21 ~ 27%p 낮게 나왔다.
- `redistributable: false` 행이 섞여 있다. 노트북 출력 · 모델 zip · csv 를 저장소에 올리지 않는다.


## 1. 환경 설정 + 파일 업로드

Colab에서 `sympy` 충돌로 `from transformers import ...`가 깨지면 런타임 > 세션 다시 시작 후 첫 셀은 건너뛰고 다음 셀부터 실행한다.
런타임에 같은 이름 파일이 이미 있으면 새로 올린 파일은 `golden (1).jsonl`로 저장되고 코드는 예전 파일을 읽는다. 새 판을 올릴 때는 런타임을 새로 시작한다.

In [ ]:
!pip install -q -U sympy transformers accelerate

In [ ]:
import os, json, hashlib, shutil, gc, math, random
from collections import Counter, defaultdict
import numpy as np

from google.colab import files as colab_files
print("① golden.jsonl 선택")
uploaded = dict(colab_files.upload())
print("② banned_terms.jsonl 선택")
uploaded.update(colab_files.upload())
print("업로드된 파일:", list(uploaded.keys()))

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

GOLDEN_SHA = sha256("golden.jsonl")
print("golden.jsonl sha256:", GOLDEN_SHA[:16], "…")
# D-176: 입력 지문을 **대조**한다. 새 판을 쓸 때는 data/derived_manifest.jsonl 의 sha 로 이 값을 바꾼다
EXPECTED_GOLDEN_SHA = "529c970556d77bd2c231526e20531294bf469f794e74a4796835a36f456eb7fe"   # v8 — 재동결 10-05 (8,383행)
EXPECTED_ROWS = (8383, 5894, 2489)   # 전체 · 학습 · 평가 (전달 10-05 §1)
EXPECTED_BANNED_SHA = "455f8d7ea5c227ffab268a33cef9bc36750b0144f073a2ed33e1f7298dc5b892"   # 사전 506종
if GOLDEN_SHA != EXPECTED_GOLDEN_SHA:
    raise SystemExit(f"🔴 golden sha 가 원장과 다르다 — 받은 {GOLDEN_SHA[:16]} · 원장 {EXPECTED_GOLDEN_SHA[:16]} (D-176). "
                     "새 판이면 EXPECTED_GOLDEN_SHA 를 derived_manifest 값으로 바꾼다")
BANNED_SHA = sha256("banned_terms.jsonl")
print("banned_terms.jsonl sha256:", BANNED_SHA[:16], "…" + ("" if BANNED_SHA == EXPECTED_BANNED_SHA else "  🟡 원장(506종)과 다르다 — 기준선 B 가 달라진다. data-sync 한 파일인지 확인"))


## 2. 데이터 로드 + eval 조건 버킷 — D-174 · D-232 · D-286 · D-301

`split` 필드를 그대로 따른다(D-174). 학습 행 고르기는 섹션 4 첫머리에서 한다(스위치를 읽어야 해서).

| 버킷 | 정의 | 기대 응답 | 재는 것 |
|---|---|---|---|
| `scored` | 확정 6종 라벨 있음 · 보조 라벨 없음 · 조건 ≠ M · D | 위반 (그 유형) | 라벨별 P/R/F1 · ROC/PR-AUC · G1 |
| `pending` | 라벨 없음 · 조건 C/A/B (화장품법 4호 등 유형 없는 위반) | 위반 (유형 미정) | 위반 탐지 재현율 |
| `aux` | 비방 · 부당비교 · 기능성화장품 오인 라벨 있음 | (확정 6종 밖) | 참고만 |
| `neg` | 라벨 없음 · 조건 L 또는 조건 없음 | 위반 없음 | **적법 문장 오탐 — 주장 있음** |
| `D` | 조건 D | 판정 대상 아님 | id `gf:` = **적법 문장 오탐 — 주장 없음** · 그 밖(해설서 위반문구)은 참고 |
| `M` | 조건 M | `low_conf` 보류 | 채점 안 함 (참고) |


In [ ]:
def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

golden = load_jsonl("golden.jsonl")
print("golden.jsonl:", len(golden), "행")

# D-232 — 인코더가 예측하는 확정 클래스 (scripts/collect.py VIOLATION_TYPES 순서)
CONFIRMED = ["질병_예방치료_표방", "건강기능식품_오인", "의약품_오인", "거짓_과장", "소비자_기만", "후기_체험기_기만"]
AUX = ["부당_비교광고", "비방광고", "기능성화장품_오인"]   # 학습 로짓 7 ~ 9번 칸 (v8 부터 9칸으로 학습한다)
# 🆕 v8.2 (D-321 · 팀장 10-06) — **확정 8종** = 기존 6 + 부당_비교광고 · 비방광고. 기능성화장품_오인 은 「편입 대기」(칸은 저장 · 문턱은 꺼 둔다)
#    🔴 CONFIRMED(6) 는 그대로 둔다 — 모델을 고르는 기준(회차 · 시드)과 2차까지의 수치가 이 6종이다. 8종은 CONF8 로 따로 든다
NEW_CONF = ["부당_비교광고", "비방광고"]
CONF8 = CONFIRMED + NEW_CONF
WAITING = ["기능성화장품_오인"]
assert CONFIRMED + AUX == CONF8 + WAITING, "저장 순서 — 기존 6 → 부당 비교 → 비방 → 기능성화장품 (팀장 10-06)"
LAW_NAMES = {"002011": "표시광고법", "013094": "식품표시광고법", "002015": "화장품법"}

def norm_text(s):
    return " ".join(s.split())

def is_inj(r):
    # 합성(주입) 행 — preprocess/inject.py 가 만든 행 (origin "injected" · id 머리 "inj:")
    return r.get("origin") == "injected" or str(r["id"]).startswith("inj:")

def group_key(r):
    # 같은 원문에서 나온 행을 한 묶음으로 — dev 분할 단위
    rid = r["id"]
    if rid.startswith("inj:"):             # 'inj:T1:hf:0:0' → 원본 승인 문구 'hf:0:0'
        return rid.split(":", 2)[2]
    return rid.split("#")[0]

def to_ho(c):
    return c.split("|")[0]

# 🆕 v8.1 — 호 → 유형 대응표. collect/statute.py 의 _HO · _MOK 와 같아야 한다(아래에서 골든의 유형 칸으로 대조한다)
_F, _C, _K = "013094:제8조제1항제", "002011:제3조제1항제", "002015:제13조제1항제"
HO_TYPE = {
    _F + "1호": "질병_예방치료_표방", _F + "2호": "의약품_오인", _F + "3호": "건강기능식품_오인", _F + "4호": "거짓_과장",
    _F + "5호": "소비자_기만", _F + "5호|다목": "후기_체험기_기만", _F + "6호": "비방광고", _F + "7호": "부당_비교광고",
    _C + "1호": "거짓_과장", _C + "2호": "소비자_기만", _C + "3호": "부당_비교광고", _C + "4호": "비방광고",
    _K + "1호": "의약품_오인", _K + "2호": "기능성화장품_오인",
}   # 여기 없는 호(화장품법 13①4 · 식품 8 ~ 10호)는 유형이 없다

def ho_key(c):
    # 학습 단위 = 호. 목이 유형을 바꾸는 자리(식품 5호 다목 = 후기)만 목까지 든다
    return c if c in HO_TYPE else c.split("|")[0]

def normalize(r):
    basis = r.get("근거") or []
    return {
        "id": r["id"], "group": group_key(r), "text": r["text"],
        "labels": r.get("labels") or [], "split": r["split"],
        "provenance": r.get("provenance"), "origin": r.get("origin"),
        "zone": r.get("구역"), "condition": r.get("조건"), "unit": r.get("unit"),
        "basis": basis, "hos": sorted({to_ho(c) for c in basis}), "hokeys": sorted({ho_key(c) for c in basis}),
        "laws": sorted({c.split(":")[0] for c in basis}),
    }

all_rows = [normalize(r) for r in golden]
unknown = {l for r in all_rows for l in r["labels"]} - set(CONFIRMED) - set(AUX)
assert not unknown, f"예상 밖 라벨: {unknown}"
# 🆕 v8.1 — 호를 유형으로 접은 것이 골든의 유형 칸과 같은가 (대응표 검증 · 호 판은 어긋나면 멈춘다)
HO_MISMATCH = [r["id"] for r in all_rows if sorted({HO_TYPE[k] for k in r["hokeys"] if k in HO_TYPE}) != sorted(r["labels"])]
print(f"호 → 유형 대조: 어긋난 행 {len(HO_MISMATCH)}" + (f" · 예 {HO_MISMATCH[:3]}" if HO_MISMATCH else " (골든의 유형 칸과 같다)"))

train_all = [r for r in all_rows if r["split"] == "train"]
eval_rows = [r for r in all_rows if r["split"] != "train"]
_got = (len(all_rows), len(train_all), len(eval_rows))
if _got != EXPECTED_ROWS:
    raise SystemExit(f"🔴 행 수가 전달 문서와 다르다 — 받은 {_got} · 기대 {EXPECTED_ROWS} (D-176)")

def train_role(r):
    # 🆕 v8 (D-296 개정 2) — 학습 행을 조건 칸으로 읽는다. 집행은 섹션 4 첫머리(스위치를 읽는다)
    #    M = 뺀다(양성도 음성도 아니다) · D · L = 음성 · C · A · B · 조건 없음 = 유형이 양성
    #    C · A · B 인데 유형이 없으면(화장품법 13①4 · 근거 결손) 유형 6종으로는 양성 자리가 없다 → "pos_untyped"
    c = r["condition"]
    if c == "M":
        return "drop_M"
    if c in ("D", "L"):
        return "neg"
    if r["labels"]:
        return "pos"
    return "pos_untyped" if c in ("C", "A", "B") else "neg"

def lawful_kind(r):
    # 🆕 v8 (D-301 · preprocess/golden.py lawful_kind 와 같은 규칙) — 적법 문장은 둘뿐이다
    if not r["labels"] and (r["condition"] or "L") == "L":
        return "주장"          # 원천이 위반 아님을 선언 (조건 L)
    if r["condition"] == "D" and r["id"].startswith("gf:"):
        return "주장없음"      # 원천이 승인한 형태 · 주장 없음 (해설서 수정문구 판)
    return None

_tc = Counter((train_role(r), r["condition"] or "없음") for r in train_all)
print("학습 행 (조건 × 역할):", {f"{c}:{ro}": n for (ro, c), n in sorted(_tc.items(), key=lambda kv: (kv[0][1], kv[0][0]))})
print(f"  M {sum(n for (ro, c), n in _tc.items() if c == 'M')} (유형 있는 것 {sum(1 for r in train_all if r['condition'] == 'M' and r['labels'])}) · "
      f"D·L {sum(n for (ro, c), n in _tc.items() if c in ('D', 'L'))} · 유형 없는 C·A·B {sum(n for (ro, c), n in _tc.items() if ro == 'pos_untyped')} · "
      f"낱말 행 {sum(1 for r in train_all if r['unit'] == '낱말')}   ← 기대: 654(541) · 612 · 178 · 327")

def eval_bucket(r):
    # 팀장 채점 기준(10-01) — C·A·B = 양성(scored + pending) · L = 음성(neg) · D = 별도 지표(D-301 내역) · M = 채점 안 함
    #    조건이 M · D 면 라벨보다 먼저 가른다(채점 밖). L 인데 라벨이 있으면 데이터가 어긋난 것이라 멈춘다.
    if r["condition"] == "M":
        return "M"
    if r["condition"] == "D":
        return "D"
    if r["condition"] == "L":
        assert not r["labels"], f"🔴 조건 L(적법)인데 라벨이 있다 — {r['id']} · {r['labels']}"
        return "neg"
    if any(l in AUX for l in r["labels"]):
        return "aux"
    if r["labels"]:
        return "scored"
    if r["condition"] in ("C", "B", "A"):
        return "pending"
    return "neg"

def eval_bucket8(r):
    # 🆕 v8.2 — 8종 기준 버킷. 조건 M · D · L · 유형 없는 위반(pending)은 6종 기준과 같다
    #    scored8 = 8종 중 하나라도 정답인 행 (6종 기준 scored 전부 + 부당 비교 · 비방이 붙은 aux 행) · 대기 = 기능성화장품_오인만 있는 행
    if r["bucket"] in ("M", "D", "neg", "pending"):
        return r["bucket"]
    return "scored8" if any(l in CONF8 for l in r["labels"]) else "대기"

for r in eval_rows:
    r["bucket"] = eval_bucket(r)
    r["bucket8"] = eval_bucket8(r)
BUCKETS = {b: [r for r in eval_rows if r["bucket"] == b] for b in ["scored", "aux", "neg", "D", "M", "pending"]}
BUCKETS["scored8"] = [r for r in eval_rows if r["bucket8"] == "scored8"]
BUCKETS["대기"] = [r for r in eval_rows if r["bucket8"] == "대기"]
assert {r["id"] for r in BUCKETS["scored"]} <= {r["id"] for r in BUCKETS["scored8"]}
print(f"train {len(train_all)} / eval {len(eval_rows)}")
print(f"  팀장 채점(6종 기준): 양성(유형 있는 위반) {len(BUCKETS['scored'])} · 유형 없는 위반(따로) {len(BUCKETS['pending'])} · 음성(L) {len(BUCKETS['neg'])} · "
      f"D(별도) {len(BUCKETS['D'])} · M(채점 안 함) {len(BUCKETS['M'])} · aux(참고) {len(BUCKETS['aux'])}")
for b, rows in BUCKETS.items():
    print(f"  {b:8s}: {len(rows):5d}  출처={dict(Counter(r['provenance'] for r in rows))}")
print(f"  🆕 8종 기준: 양성 {len(BUCKETS['scored8'])} (6종 기준 scored {len(BUCKETS['scored'])} + 부당 비교 · 비방이 붙은 행 {len(BUCKETS['scored8']) - len(BUCKETS['scored'])}) · "
      f"편입 대기(기능성화장품만) {len(BUCKETS['대기'])} · 유형 없는 위반 {len(BUCKETS['pending'])}   ← 기대: 유형 없는 위반 131 (채점에서 빼고 따로 센다)")
print("  평가 정답 수(조건 M 제외):", {l: sum(l in r["labels"] for r in eval_rows if r["bucket"] != "M") for l in NEW_CONF + WAITING})
_lk = Counter(lawful_kind(r) for r in eval_rows)
print(f"  적법 문장(D-301): 주장 있음 {_lk['주장']} · 주장 없음 {_lk['주장없음']} · 해설서 위반문구 D(적법 아님) {len(BUCKETS['D']) - _lk['주장없음']}")

## 3. 실험 스위치 — 이 셀만 바꾸고 「섹션 3부터 끝까지」 다시 실행

| 스위치 | 기본값 | 뜻 |
|---|---|---|
| `LABEL_UNIT` | `"type"` | 학습 단위 — `"type"`(유형 · 후보) / `"ho"`(호로 학습해 유형으로 접는다 · 비교용 · 저장 안 함) |
| `DEV_SCORE_INJECTED` | `False` | 🆕 v8.3 dev 채점에 주입(합성) 행을 넣을지. `False` = 실제 문구만으로 모델을 고르고 문턱을 잡는다 · `True` = v8.2 까지의 dev |
| `DEV_STRATIFY_NEW` | `True` | 🆕 v8.3 dev 를 뗄 때 새 확정 2종(부당 비교 · 비방)도 층화한다 · `False` = 6종만(v8.2 까지) |
| `INJ_WEIGHT` | `1.0` | 🆕 v8.3 주입(합성) 학습 행의 손실 가중치. 실험 `0.3` |
| `MAX_EPOCHS` | `6` | epoch 상한. 실험 `8` |
| `USE_WORD_ROWS` | `False` | 🆕 사례집 **낱말 행(327)** 을 학습에 넣을지. `False` 면 뺀 판 — 둘을 dev 로 비교한다 |
| `DROP_UNTYPED_POS` | `True` | 🆕 유형 없는 위반 행(조건 C·A·B · 178)을 학습에서 뺀다. `False` 면 음성으로 넣는다(v7 방식 · 권하지 않는다) |
| `REASON_MODE` | `"all"` | 이유 구역 처리 — `all`(D-234) / `weighted` / `exclude` / `label_mask` / `mask_nonclaim`. v8 기본은 `all` |
| `USE_AUX_LOGITS` | `True` | 🔴 v8.2 는 늘 `True` — 7 ~ 9번 칸(부당 비교 · 비방 · 기능성화장품)을 학습해야 9칸으로 저장한다 |
| `USE_INJECTED` | `True` | 주입 행(D-25)을 학습에 넣을지 |
| `DEDUP` | `True` | train 안 같은 문구 병합 |
| `SEEDS` | `[42, 43, 44]` | 시드마다 학습 한 번 |
| `THRESHOLD_MODE` | `"robust"` | 🆕 v9 문턱 규칙 — `"robust"`(시드 3개의 dev F1 평균 최대) / `"best_seed"`(고른 시드 하나 · v8.3 R0 과 같다). 🔴 test 전에 정한다 |
| `FINAL_TEST` | `False` | 🔴 test 채점은 최종 후보 1회만 (D-175). A 회차 뒤 이것만 `True` 로 바꾸고 섹션 3부터 다시 실행 — 다시 학습하지 않는다 |
| `SAVE_THIS_RUN` | `True` | 모델 zip 저장 — 로컬 판정 분포(`run_judge_dist.py`)에 쓴다 |
| `EXTRA_TRAIN_FILES` | `[]` | 🔄 추가 학습 파일 — 이번 3차 측정은 없이 돌린다 |

조건 M 은 스위치가 없다 — 언제나 뺀다(D-296 개정 2). 확정 8종 · 편입 대기도 스위치가 없다(D-321).


In [ ]:
BACKBONE = "kcbert"                  # "kcbert"(D-70) / "kcelectra"(D-84)
LABEL_UNIT = "type"                  # 학습 단위. "type"(유형 · 후보 — 팀장 10-06) / "ho"(호로 학습 → 유형으로 접어 채점 · 비교용 · 저장 안 함)
USE_WORD_ROWS = False                # 사례집 낱말 행(327). 뺀 판이 후보다 (dev 10-05 · 팀장 동의 10-06)
DROP_UNTYPED_POS = True              # 유형 없는 위반 행(조건 C·A·B · 178 · 근거 빈 5 포함)을 학습에서 뺀다 (팀장 확인 10-06)
# 🆕 v8.3 — 개선 실험 스위치. 한 번에 하나만 바꾼다 (머리말 실행 순서)
DEV_SCORE_INJECTED = False           # dev 채점에 주입(합성) 행을 넣을지. False = 실제 문구만으로 회차 · 시드를 고르고 문턱을 잡는다 · True = v8.2 까지
DEV_STRATIFY_NEW = True              # dev 를 뗄 때 새 확정 2종(부당 비교 · 비방)도 층화한다 · False = 6종만(v8.2 까지)
INJ_WEIGHT = 1.0                     # 주입(합성) 학습 행의 손실 가중치. 1.0 = 기준(R0) · 실험 0.3 (R1)
REASON_MODE = "all"                  # "all"(D-234 · v8 기본) / "weighted" / "exclude" / "label_mask" / "mask_nonclaim"
REASON_WEIGHT = 0.4                  # weighted일 때만
REASON_MASK_LABELS = ["거짓_과장", "소비자_기만"]   # label_mask · mask_nonclaim일 때
# v7.3 — mask_nonclaim: 이유 구역 중 짧고(NONCLAIM_MAXLEN 미만) 주장 어휘가 없는 행만 마스킹
NONCLAIM_MAXLEN = 25
NONCLAIM_CUE = (r"[!?~]|효과|효능|개선|회복|예방|치료|완치|최고|최초|유일|1위|No\.?1|넘버원|보장|무료|할인|특가|최저|저렴|혁명|명품|"
                r"프리미엄|완벽|확실|안전|인증|특허|수상|추천|인기|만족|보증|증명|입증|검증|공인|대박|파격|절약|혜택|수익|확정|평생|"
                r"100%|\d+\s*%|배|만원|억|원\b|싸|좋|최|초|강력|탁월|우수|뛰어|놀라|기적|비밀|특별|독점|단독|전문|정품|천연|무첨가|"
                r"친환경|역세권|대단지|분양|임대")
USE_AUX_LOGITS = True
USE_INJECTED = True
DEDUP = True

SEEDS = [42, 43, 44]
DEV_FRAC, DEV_SEED = 0.15, 42        # dev 분할은 시드와 무관하게 고정
MAX_EPOCHS, PATIENCE = 6, 2
BATCH, LR, WARMUP = 32, 2e-5, 0.1
USE_FOCAL_LOSS, FOCAL_GAMMA = True, 1.5
POS_WEIGHT_CAP, POS_WEIGHT_SQRT = 5, True

SAVE_THIS_RUN = True                # 모델 zip 저장 (로컬 판정 분포에 쓴다)
FINAL_TEST = False                   # v7.4 — 🔴 test 채점은 최종 후보 1회만 (D-175). 실험 비교는 dev 로
EXTRA_HOLDOUT_FRAC = 0.2             # v7.4 — 추가 파일마다 같은 20% 를 떼어 홀드아웃 지표로 잰다(어느 실행에서도 학습 안 함)
THRESHOLD_MODE = "robust"            # "best_seed"(저장 모델 자신의 dev threshold · 기본) / "robust"(시드 평균 F1 최대 · 참고용)

# v7.1 — train에만 붙일 추가 jsonl (실험 전용 · 채택은 팀장 판정). None이면 v7과 같다
EXTRA_TRAIN_FILES = []              # 🔄 v8 — 3차 골든 첫 측정은 추가 파일 없이 (10-05). 붙이려면 ["proto_inj4c.jsonl", "proto_negd4.jsonl"] · negd4 는 새 분할로 다시 만든다
EVAL_ONLY_FILES = []                # 학습하지 않고 홀드아웃 지표만 잴 파일 — 기준 실행에서 쓴다

BACKBONE_MODEL_MAP = {"kcbert": "beomi/kcbert-base", "kcelectra": "beomi/KcELECTRA-base"}
MODEL_NAME = BACKBONE_MODEL_MAP[BACKBONE]
assert LABEL_UNIT in ("type", "ho"), LABEL_UNIT
if LABEL_UNIT == "ho" and HO_MISMATCH:
    raise SystemExit(f"🔴 호 → 유형 대응표가 골든과 다르다 — {len(HO_MISMATCH)}행 · 예 {HO_MISMATCH[:5]}. HO_TYPE 을 collect/statute.py 와 맞춘다")
UNTYPED = "유형없음"                 # 호 판에서만 — 유형이 없는 호(화장품법 13①4 등)가 접히는 칸
# 채점 공간(유형). 호 판은 학습 로짓(호)을 이 칸들로 접는다 — 섹션 4 의 fold()
LABEL_LIST = (CONFIRMED + AUX + [UNTYPED]) if LABEL_UNIT == "ho" else CONFIRMED + (AUX if USE_AUX_LOGITS else [])
LABEL2ID = {l: i for i, l in enumerate(LABEL_LIST)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}
N_CONF = len(CONFIRMED)              # 앞 6개 — 🔴 모델 선택(조기 종료 · 가장 좋은 시드)과 6종 기준 표는 이 6칸으로만 잰다 (팀장 10-06)
# 🆕 v8.2 (D-321) — 확정 8종 · 편입 대기
if not USE_AUX_LOGITS:
    raise SystemExit("🔴 v8.2 는 9칸(확정 8 + 편입 대기 1)으로 학습 · 저장한다 — USE_AUX_LOGITS 를 끄지 않는다 (D-321)")
N8 = len(CONF8)                      # 앞 8개가 확정 클래스 (문턱을 dev 에서 고른다)
assert LABEL_LIST[:N8] == CONF8 and LABEL_LIST[N8] == WAITING[0], LABEL_LIST
IDX_WAIT = LABEL2ID[WAITING[0]]      # 9번 칸 — 기능성화장품_오인 (편입 대기)
TH_OFF = 1.01                        # 편입 대기 칸의 문턱 — 1 초과라 `확률 >= 문턱` 이 참이 될 수 없다 (꺼 둠 · 팀장 10-06)
WAIT_REF_TH = 0.5                    # 편입 대기 칸을 참고로 잴 때의 문턱 (팀장 10-06 — test 수치는 0.5 기준 참고값)

reason_tag = {"weighted": f"weighted{REASON_WEIGHT}",
              "label_mask": "mask(" + ",".join(REASON_MASK_LABELS) + ")",
              "mask_nonclaim": f"mask_nonclaim(<{NONCLAIM_MAXLEN})"}.get(REASON_MODE, REASON_MODE)
loss_tag = (f"focal(g={FOCAL_GAMMA},cap={POS_WEIGHT_CAP}{',sqrt' if POS_WEIGHT_SQRT else ''})"
            if USE_FOCAL_LOSS else f"bce(cap={POS_WEIGHT_CAP})")
def _flags(word=True, untyped_drop=True, unit="type", conf8=False, dev_inj=True, dev_new=False, inj_w=1.0, ep=6, inj=None):
    # 실험 키의 뒷부분 — 기본 스위치(낱말 포함 · 유형 없는 위반 제외)면 "|v8" 로 끝난다
    # 🆕 v8.2 — conf8=True 면 "|확정8" 이 붙는다. v8 · v8.1 로그(6종 저장판)의 키는 그대로 읽힌다
    # 🆕 v8.3 — dev 정의 · 주입 가중치 · epoch 상한이 뒤에 붙는다. 기본 인자는 v8.2 까지의 값이라 옛 키가 그대로 나온다
    inj = USE_INJECTED if inj is None else inj
    f = ("" if USE_AUX_LOGITS else "|noaux") + ("" if inj else "|noinj") + ("" if DEDUP else "|nodedup")
    for _f in EXTRA_TRAIN_FILES:
        f += f"|+{os.path.splitext(os.path.basename(_f))[0]}"
    return (f + "|v8" + ("" if word else "|낱말제외") + ("" if untyped_drop else "|유형없음=음성") + ("" if unit == "type" else "|호학습")
            + ("|확정8" if conf8 else "") + ("" if dev_inj else "|dev실제") + ("|dev새2종" if dev_new else "")
            + ("" if inj_w == 1.0 else f"|주입w{inj_w:g}") + ("" if ep == 6 else f"|ep{ep}"))
assert not set(EXTRA_TRAIN_FILES) & set(EVAL_ONLY_FILES), "같은 파일을 학습 · 평가 전용에 함께 둘 수 없다"
EXTRA_SHA = {}
for _f in list(EXTRA_TRAIN_FILES) + list(EVAL_ONLY_FILES):
    if not os.path.exists(_f):
        print(f"③ {_f} 선택")
        colab_files.upload()
    EXTRA_SHA[_f] = sha256(_f)
    print(f"{_f} sha256: {EXTRA_SHA[_f][:16]}…" + ("  (평가 전용)" if _f in EVAL_ONLY_FILES else ""))
DEV_LEGACY = DEV_SCORE_INJECTED and not DEV_STRATIFY_NEW   # v8.2 까지의 dev 정의인가 (합성 포함 · 6종 층화)
_dv = dict(dev_inj=DEV_SCORE_INJECTED, dev_new=DEV_STRATIFY_NEW)
flags = _flags(USE_WORD_ROWS, DROP_UNTYPED_POS, LABEL_UNIT, conf8=True, inj_w=INJ_WEIGHT, ep=MAX_EPOCHS, **_dv)
_th_tag = "|th=robust" if THRESHOLD_MODE == "robust" else ""
exp_key = f"{BACKBONE}|이유={reason_tag}|{loss_tag}{flags}{_th_tag}"
# 🆕 v8.2 — 같은 설정의 **6종 저장판**(v8 · v8.1 로그). 학습이 같으므로 6종 dev 수치가 같아야 한다 → 섹션 9 재현 확인
PREV_KEY = f"{BACKBONE}|이유={reason_tag}|{loss_tag}{_flags(USE_WORD_ROWS, DROP_UNTYPED_POS, LABEL_UNIT)}{_th_tag}"
# 비교 기준(사전 고정 기준 G1 ~ G3 · B2 · 6종 dev) — 호 판은 **같은 설정의 유형 판** · 그 밖은 all · 낱말 포함(1회차)
#   호 판의 기준은 8종 저장판 유형 실행이고, 로그에 없으면 섹션 9 가 6종 저장판(2회차)으로 바꿔 견준다 — 6종 dev 수치는 같다
# 🔄 v8.3 — 비교 기준:
#   · 호 판 → 같은 설정의 유형 판
#   · 유형 판 · 새 dev 정의 → **같은 dev · 같은 낱말 설정의 기본 학습**(주입 가중치 1.0 · epoch 6 · 주입 포함) = R0
#   · 유형 판 · 옛 dev 정의 → v8.2 와 같다(all · 낱말 포함 1회차)
if LABEL_UNIT == "ho":
    BASE_KEY = f"{BACKBONE}|이유={reason_tag}|{loss_tag}{_flags(USE_WORD_ROWS, DROP_UNTYPED_POS, conf8=True, inj_w=INJ_WEIGHT, ep=MAX_EPOCHS, **_dv)}{_th_tag}"
elif DEV_LEGACY:
    BASE_KEY = f"{BACKBONE}|이유=all|{loss_tag}{_flags()}{_th_tag}"
else:
    BASE_KEY = f"{BACKBONE}|이유=all|{loss_tag}{_flags(USE_WORD_ROWS, DROP_UNTYPED_POS, conf8=True, inj=True, **_dv)}{_th_tag}"
BASE_KEY_ALT = (f"{BACKBONE}|이유={reason_tag}|{loss_tag}{_flags(USE_WORD_ROWS, DROP_UNTYPED_POS)}{_th_tag}"
                if LABEL_UNIT == "ho" and DEV_LEGACY and INJ_WEIGHT == 1.0 and MAX_EPOCHS == 6 else None)
# 🆕 v9 — 문턱 규칙만 바꾼 판은 **같은 학습의 best_seed 판(R0)** 과 견준다 (키에서 문턱 꼬리표만 뗀 것)
if THRESHOLD_MODE == "robust" and LABEL_UNIT == "type":
    BASE_KEY = BASE_KEY.replace("|th=robust", "")
RUN_NAME = (f"copylane-encoder-{BACKBONE}-v9-{REASON_MODE}" + ("" if USE_WORD_ROWS else "-낱말제외")
            + ("" if DROP_UNTYPED_POS else "-유형없음음성") + ("" if LABEL_UNIT == "type" else "-호학습") + "-확정8"
            + ("" if DEV_SCORE_INJECTED else "-dev실제") + ("-dev새2종" if DEV_STRATIFY_NEW else "")
            + ("" if INJ_WEIGHT == 1.0 else f"-주입w{INJ_WEIGHT:g}") + ("" if MAX_EPOCHS == 6 else f"-ep{MAX_EPOCHS}")
            + ("" if USE_INJECTED else "-noinj") + ("-문턱robust" if THRESHOLD_MODE == "robust" else ""))   # 저장 zip 이름 — 실험마다 다르다
print("exp_key =", exp_key)
print("저장 이름 =", RUN_NAME + ".zip", "· 비교 기준 =", "이 실행" if exp_key == BASE_KEY else BASE_KEY)
print("LABEL_LIST =", LABEL_LIST)
print(f"확정 8종 = 앞 {N8}칸 (모델 선택은 앞 {N_CONF}칸) · 편입 대기 = {WAITING} (문턱 {TH_OFF} — 꺼 둠)")
print(f"dev 정의 = {'합성 포함' if DEV_SCORE_INJECTED else '실제 문구만'} · {'8종 층화' if DEV_STRATIFY_NEW else '6종 층화'}"
      f"{'  (v8.2 까지와 같다)' if DEV_LEGACY else '  🆕 v8.3 — v8 ~ v8.2 실행과 견주지 않는다'} · 주입 가중치 {INJ_WEIGHT:g} · epoch 상한 {MAX_EPOCHS}")

LOG_PATH = "/content/drive/MyDrive/copylane_v8_experiment_log.json"   # v8 — 평가셋이 달라 v7 로그와 나눈다
if "EXPERIMENT_LOG" not in globals():
    EXPERIMENT_LOG = {}
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        if os.path.exists(LOG_PATH):
            with open(LOG_PATH, encoding="utf-8") as f:
                EXPERIMENT_LOG = json.load(f)
            print(f"Drive에서 이전 실험 {len(EXPERIMENT_LOG)}건 복원:", list(EXPERIMENT_LOG))
    except Exception as e:
        print("이전 실험 로그 복원 안 함:", e)

## 4. 학습 행 고르기 + dev 분리 + 학습셋 구성

🆕 v8.3 — dev 채점 행에서 **주입(합성) 행을 뺀다**(`DEV_SCORE_INJECTED = False`) · dev 를 뗄 때 **새 확정 2종도 층화**한다(`DEV_STRATIFY_NEW = True`). dev 묶음에 든 주입 행은 학습에도 쓰지 않는다 — 원본 문구가 dev 에 있기 때문이다(묶음 단위 분리).


0. 🆕 **학습 행 고르기 (D-296 개정 2)** — 조건 M 을 빼고 · D · L 을 음성으로 · 유형 없는 위반과 (스위치에 따라) 낱말 행을 뺀다. 표가 머리말의 기대 수와 같은지 본다.
1. **dev 분리** — train의 묶음(문서 · 원본 문구) 단위로, 확정 6종마다 「이유가 아닌 양성」의 약 15%가 dev에 가도록 층화해서 뽑는다. 음성 묶음도 15%를 넣는다.
2. dev 채점은 그 묶음의 **이유가 아닌 행**으로만 한다. dev 묶음의 이유 행은 train에서도 뺀다(같은 문서가 양쪽에 서지 않게).
3. `DEDUP` — train 안 같은 문구(같은 구역 종류끼리)를 한 행으로 합치고 라벨은 합집합으로 둔다. dev 와 같은 문구는 train에서 뺀다.
4. `REASON_MODE`에 따라 행 가중치(`sample_weight`)와 라벨 마스크(`loss_mask`)를 정한다.


In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

def is_reason(r):
    return r["provenance"] == "ftc_decisions_body" and r["zone"] == "이유"

def devable(r):
    # dev 채점 행 — 이유 구역은 뺀다(v7 그대로). 🆕 v8: 낱말 행 · 유형 없는 위반도 뺀다
    #   → dev 가 스위치(USE_WORD_ROWS · DROP_UNTYPED_POS)와 무관하게 **같은 행**이라 판끼리 바로 비교된다. 문장 단위 모델은 문장으로 잰다
    # 🆕 v8.3: DEV_SCORE_INJECTED=False 면 주입(합성) 행도 뺀다 — v8.2 dev 463행 중 140행이 주입이었고 모델이 전부 맞혀 수치가 부풀었다
    return (not is_reason(r) and r["unit"] != "낱말" and r.get("role") != "pos_untyped"
            and (DEV_SCORE_INJECTED or not is_inj(r)))

# ── 0) 🆕 v8 학습 행 고르기 (D-296 개정 2) — M 제외 · D · L 음성 · 유형 없는 위반 · 낱말 행은 스위치
#    train_use = 조건 M 만 뺀 전부(dev 분리의 모집단 — 스위치와 무관). 스위치는 아래 2) train 풀에서 건다
train_use, ROLE_COUNTS = [], Counter()
for r in train_all:
    role = train_role(r)
    shown = role
    if role == "pos_untyped":
        if LABEL_UNIT == "ho" and r["hokeys"]:
            shown = "pos_호만"       # 🆕 v8.1 — 호 판은 유형 없는 호도 그 호의 양성으로 쓴다
        else:
            shown = "drop_유형없음" if DROP_UNTYPED_POS else "neg"
    if r["unit"] == "낱말" and not USE_WORD_ROWS and not shown.startswith("drop"):
        shown = "drop_낱말"
    ROLE_COUNTS[f"{shown}|{r['condition'] or '없음'}"] += 1
    if role == "drop_M":
        continue
    kind = "drop" if shown.startswith("drop") else ("pos" if shown.startswith("pos") else "neg")   # 이 판에서의 쓰임
    train_use.append(dict(r, role=role, kind=kind, labels=list(r["labels"]) if role == "pos" else []))
_by = lambda p: sum(n for k, n in ROLE_COUNTS.items() if k.startswith(p))
print(f"학습 행 고르기: 전체 {len(train_all)} → 쓰는 행 {_by('pos') + _by('neg|')}  (양성 {_by('pos')} · 음성 {_by('neg|')})"
      + (f" · 그중 유형 없는 호를 양성으로 쓰는 행 {_by('pos_호만')}" if LABEL_UNIT == "ho" else ""))
print(f"  뺀 행 — 조건 M {_by('drop_M')} · 유형 없는 위반 {_by('drop_유형없음')} · 낱말 {_by('drop_낱말')}")
print("  내역:", dict(sorted(ROLE_COUNTS.items())))
assert not any(r["condition"] == "M" for r in train_use), "조건 M 이 학습에 남았다 (D-296 개정 2)"
assert not any(r["labels"] for r in train_use if r["condition"] in ("D", "L")), "조건 D · L 행에 유형이 남았다"

# ── 1) 묶음 단위 층화 dev 분리
groups = defaultdict(list)
for r in train_use:
    groups[r["group"]].append(r)
rng = random.Random(DEV_SEED)
gkeys = sorted(groups)
rng.shuffle(gkeys)

STRAT = CONFIRMED + (NEW_CONF if DEV_STRATIFY_NEW else [])   # 🆕 v8.3 — 층화하는 유형. 새 확정 2종을 넣으면 실제 부당 비교 · 비방 행이 dev 에 들어온다
pos_total = Counter(l for r in train_use if devable(r) for l in r["labels"] if l in STRAT)
target = {l: max(1, math.ceil(DEV_FRAC * pos_total[l])) for l in STRAT}
print("dev 층화 모집단(채점할 수 있는 양성 행) → 목표:", {l: f"{pos_total[l]} → {target[l]}" for l in STRAT})
neg_groups = [g for g in gkeys if any(devable(r) and not r["labels"] for r in groups[g])]
neg_target = math.ceil(DEV_FRAC * len(neg_groups))
got, neg_got, dev_groups = Counter(), 0, set()
for g in gkeys:
    labs = Counter(l for r in groups[g] if devable(r) for l in r["labels"] if l in STRAT)
    need = any(got[l] < target[l] for l in labs)
    is_neg = g in neg_groups and not labs
    if need or (is_neg and neg_got < neg_target):
        dev_groups.add(g)
        got.update(labs)
        neg_got += is_neg

dev_rows = [r for g in dev_groups for r in groups[g] if devable(r)]
dev_texts = {norm_text(r["text"]) for r in dev_rows}
print(f"dev: 묶음 {len(dev_groups)}개 · 채점 행 {len(dev_rows)} (음성 {sum(1 for r in dev_rows if not r['labels'])})")
print("dev 양성:", {l: sum(l in r['labels'] for r in dev_rows) for l in CONFIRMED})
print("dev 음성 조건:", dict(Counter(r["condition"] or "없음" for r in dev_rows if not r["labels"])))
# 🆕 v8.2 — dev 는 6종으로 층화한 그대로다(2회차와 같은 행). 새 확정 2종의 dev 양성은 우연히 들어온 수다 → 수를 먼저 본다
DEV_NEW_N = {l: (sum(l in r["labels"] for r in dev_rows), sum(l in r["labels"] for r in dev_rows if not is_inj(r))) for l in NEW_CONF + WAITING}
print("dev 양성 — 새 확정 2종 · 편입 대기 (전체 / 주입 제외):", {l: f"{a} / {b}" for l, (a, b) in DEV_NEW_N.items()})
for l in NEW_CONF:
    if DEV_NEW_N[l][0] < 10:
        print(f"  🔴 {l}: dev 양성 {DEV_NEW_N[l][0]}건 — 10건 미만이라 문턱을 고르지 못한다(0.5 로 둔다). 팀장에게 알린다")
    elif DEV_NEW_N[l][0] < 30:
        print(f"  🟡 {l}: dev 양성 {DEV_NEW_N[l][0]}건 — 문턱은 고르지만 30건 미만이라 dev 수치는 「측정 불가」로 적는다 (D-40)")
# 🆕 v8.1 — dev 묶음에 든 유형 없는 위반. 어느 판도 학습하지 않는다(묶음째 빠진다) → 탐지율을 참고로 잰다
DEV_UNTYPED = [r for g in dev_groups for r in groups[g] if r["role"] == "pos_untyped" and not is_reason(r) and r["unit"] != "낱말"]
print(f"dev 묶음의 유형 없는 위반: {len(DEV_UNTYPED)}행 (참고 지표 · 학습 안 함)")
# 🆕 v8.3 — dev 묶음에 든 주입(합성) 행. 채점에서 뺐고 학습에도 쓰지 않는다(묶음째 빠진다) → 탐지율을 참고로만 찍는다
DEV_INJ = [] if DEV_SCORE_INJECTED else [r for g in dev_groups for r in groups[g] if is_inj(r) and r["unit"] != "낱말" and not is_reason(r)]
DEV_SHA = hashlib.sha256("|".join(sorted(r["id"] for r in dev_rows)).encode()).hexdigest()[:16]   # 같은 dev 행으로 잰 실행끼리만 견준다
print(f"dev 묶음의 주입 행: {len(DEV_INJ)}행 (채점 밖 · 학습 안 함 · 참고) · dev 행 지문 {DEV_SHA}")
if not DEV_SCORE_INJECTED:
    assert not any(is_inj(r) for r in dev_rows), "dev 채점 행에 주입 행이 남았다"

# ── 2) train 풀
# 스위치(유형 없는 위반 · 낱말 행 · 학습 단위)는 0) 에서 정한 kind 에 들어 있다 — "drop" 만 뺀다
pool = [r for r in train_use if r["group"] not in dev_groups and r["kind"] != "drop"]
if not USE_INJECTED:
    pool = [r for r in pool if r["origin"] != "injected"]
before = len(pool)
# v7.4 — test 와 같은 문구는 train 에서 빼지 않는다(D-174 · 누수는 물질화가 평가 쪽에서 뺀다). dev 와 같은 문구만 뺀다
pool = [r for r in pool if norm_text(r["text"]) not in dev_texts]
print(f"train 풀: {before} → dev와 같은 문구 제거 후 {len(pool)}")

# ── 3) 같은 문구 병합 (이유/비이유는 따로 — 마스크 규칙이 다르므로)
if DEDUP:
    merged = {}
    for r in pool:
        k = (norm_text(r["text"]), is_reason(r))
        if k in merged:
            m = merged[k]
            m["labels"] = sorted(set(m["labels"]) | set(r["labels"]))
            if r["kind"] == "pos":      # 🆕 v8.1 — 호 라벨도 합집합 · 양성이 하나라도 있으면 양성
                m["hokeys"] = sorted(set(m["hokeys"]) | set(r["hokeys"]))
                m["kind"] = "pos"
        else:
            merged[k] = dict(r, labels=list(r["labels"]), hokeys=list(r["hokeys"]) if r["kind"] == "pos" else [])
    pool = list(merged.values())
    print(f"같은 문구 병합 후 {len(pool)}행")

# ── 3.5) v7.1 추가 train 파일 — dev · test · 기존 train과 같은 문구는 버린다
extra_rows, extra_holdout = [], {}   # extra_holdout: 파일 → 홀드아웃 행 (학습 안 함)
_have = {norm_text(r["text"]) for r in pool}
for _f in sorted(set(EXTRA_TRAIN_FILES) | set(EVAL_ONLY_FILES)):
    _all = load_jsonl(_f)
    # family 별 · id 순 · 시드 고정으로 같은 EXTRA_HOLDOUT_FRAC 를 뗀다 — 학습 여부와 무관하게 같은 행이다
    _fam = defaultdict(list)
    for r in _all:
        _fam[r.get("family", "-")].append(r)
    _hold, _hrng = [], random.Random(DEV_SEED)
    for fam, rs in sorted(_fam.items()):
        rs = sorted(rs, key=lambda r: r["id"])
        _hrng.shuffle(rs)
        _hold += rs[:int(round(EXTRA_HOLDOUT_FRAC * len(rs)))]
    extra_holdout[_f] = [normalize(r) for r in _hold]
    if _f not in EXTRA_TRAIN_FILES:
        print(f"평가 전용 {_f}: 홀드아웃 {len(_hold)}행 (학습 안 함)")
        continue
    _hid, _n0 = {r["id"] for r in _hold}, len(extra_rows)
    for r in _all:
        if r["id"] in _hid:
            continue
        x = normalize(r)
        t = norm_text(x["text"])
        if t in dev_texts or t in _have:
            continue
        assert set(x["labels"]) <= set(LABEL_LIST), x["labels"]
        x["kind"] = "pos" if x["labels"] else "neg"
        if LABEL_UNIT == "ho" and x["labels"] and not x["hokeys"]:
            raise SystemExit(f"🔴 {_f}: 근거(호)가 없는 양성 행이다 — 호 판에는 붙일 수 없다 ({x['id']})")
        _have.add(t)
        extra_rows.append(x)
    print(f"추가 train {_f}: {len(extra_rows) - _n0}행 · 홀드아웃 {len(_hold)}행 · 라벨 {dict(Counter(l for r in extra_rows[_n0:] for l in r['labels']))}")
pool = pool + extra_rows

# ── 4) REASON_MODE
if REASON_MODE == "exclude":
    pool = [r for r in pool if not is_reason(r)]
train_rows = pool
train_weights = [(REASON_WEIGHT if (REASON_MODE == "weighted" and is_reason(r)) else 1.0) * (INJ_WEIGHT if is_inj(r) else 1.0)
                 for r in train_rows]   # 🆕 v8.3 — 주입(합성) 행의 손실 가중치
print(f"학습 행 중 주입 {sum(map(is_inj, train_rows))}행 · 손실 가중치 {INJ_WEIGHT:g}")
# 🆕 v8.1 — 학습 로짓의 단위. 유형 판: 유형(확정 6 + 보조) · 호 판: 학습에 양성이 있는 호 전부
if LABEL_UNIT == "ho":
    TRAIN_LABELS = sorted({k for r in train_use if r["role"] in ("pos", "pos_untyped") for k in r["hokeys"]})
    TRAIN_TYPE = [HO_TYPE.get(k, UNTYPED) for k in TRAIN_LABELS]     # 로짓마다 접히는 유형
else:
    TRAIN_LABELS, TRAIN_TYPE = list(LABEL_LIST), list(LABEL_LIST)
TRAIN2ID = {k: i for i, k in enumerate(TRAIN_LABELS)}
FOLD_IDX = [[i for i, t in enumerate(TRAIN_TYPE) if t == l] for l in LABEL_LIST]

def fold(p):
    # 학습 로짓 공간(행, 호) → 채점 공간(행, 유형 = LABEL_LIST). 한 유형에 호가 여럿이면 가장 큰 값. 유형 판에서는 그대로다
    if LABEL_UNIT != "ho":
        return p
    out = np.zeros((p.shape[0], len(LABEL_LIST)), dtype=np.float32)
    for j, idx in enumerate(FOLD_IDX):
        if idx:
            out[:, j] = p[:, idx].max(axis=1)
    return out

mask_idx = [i for i, t in enumerate(TRAIN_TYPE) if t in REASON_MASK_LABELS]
import re as _re
_cue = _re.compile(NONCLAIM_CUE)
def is_nonclaim_reason(r):
    t = r["text"].strip()
    return is_reason(r) and len(t) < NONCLAIM_MAXLEN and not _cue.search(t)
if REASON_MODE == "mask_nonclaim":
    _nc = [r for r in train_rows if is_nonclaim_reason(r)]
    print(f"mask_nonclaim: 이유 {sum(map(is_reason, train_rows))}행 중 {len(_nc)}행의 {REASON_MASK_LABELS} 손실을 끈다 · 예: {[r['text'] for r in _nc[:6]]}")

def to_multihot(labels):
    v = np.zeros(len(LABEL_LIST), dtype=np.float32)
    for l in labels:
        if l in LABEL2ID:
            v[LABEL2ID[l]] = 1.0
    return v

def to_target(r):
    # 학습 정답 — 유형 판은 유형 칸 · 호 판은 근거의 호(양성 행만)
    if LABEL_UNIT != "ho":
        return to_multihot(r["labels"])
    v = np.zeros(len(TRAIN_LABELS), dtype=np.float32)
    if r.get("kind") == "pos":
        for k in r["hokeys"]:
            v[TRAIN2ID[k]] = 1.0
    return v

def row_mask(r):
    m = np.ones(len(TRAIN_LABELS), dtype=np.float32)
    if (REASON_MODE == "label_mask" and is_reason(r)) or (REASON_MODE == "mask_nonclaim" and is_nonclaim_reason(r)):
        m[mask_idx] = 0.0
    return m

w = np.array(train_weights)
Y = np.stack([to_target(r) for r in train_rows])
M = np.stack([row_mask(r) for r in train_rows])
WM = w[:, None] * M
TRAIN_BY_COND = dict(sorted(Counter(r["condition"] or "없음" for r in train_rows).items()))   # 저장 정보 (전달 10-05 §4-3)
print(f"\nREASON_MODE={REASON_MODE}: train {len(train_rows)}행 (이유 {sum(map(is_reason, train_rows))}행) · 조건별 {TRAIN_BY_COND}")
if INJ_WEIGHT != 1.0 or REASON_MODE == "weighted":
    print("(아래 표의 학습 행 · 양성 수는 손실 가중치를 곱한 값이다 — 행 수 자체는 위의 train 행 수)")
print(f"{'라벨':20s} {'학습 행':>8s} {'양성':>7s} {'양성 비율':>8s}")
for i, l in enumerate(TRAIN_LABELS):
    d, p = WM[:, i].sum(), (WM[:, i] * Y[:, i]).sum()
    tag = f"  → {TRAIN_TYPE[i]}" if LABEL_UNIT == "ho" else ("" if i < N_CONF else ("  (🆕 확정 · D-321)" if i < N8 else "  (편입 대기)"))
    print(f"  {l:28s} {d:8.0f} {p:7.0f} {p / max(d, 1e-6) * 100:7.1f}%{tag}")
# 🆕 v8.2 — 새 확정 2종의 학습 양성 중 합성(주입) 비율 (팀장 10-06 — 부당 비교는 60% 가량이 합성)
TRAIN_INJ = {l: [sum(l in r["labels"] for r in train_rows), sum(l in r["labels"] for r in train_rows if is_inj(r))] for l in NEW_CONF + WAITING}
print("학습 양성 중 주입(합성):", {l: f"{b}/{a} ({b / max(a, 1) * 100:.0f}%)" for l, (a, b) in TRAIN_INJ.items()})

# ── 토크나이즈
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_ds(rows, weights=None, masks=None):
    ds = Dataset.from_dict({
        "text": [r["text"] for r in rows],
        "labels": [to_target(r) for r in rows],
        "sample_weight": list(weights) if weights is not None else [1.0] * len(rows),
        "loss_mask": list(masks) if masks is not None else [np.ones(len(TRAIN_LABELS), dtype=np.float32)] * len(rows),
    })
    ds = ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=128), batched=True)
    ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels", "sample_weight", "loss_mask"])
    return ds

train_ds = to_ds(train_rows, train_weights, M)
dev_ds = to_ds(dev_rows)
print("train_ds:", len(train_ds), "/ dev_ds:", len(dev_ds), "· 학습 로짓", len(TRAIN_LABELS), "칸 →", "유형으로 접어 채점" if LABEL_UNIT == "ho" else "유형 그대로")
if LABEL_UNIT == "ho":   # 접은 정답이 유형 정답과 같아야 한다 — 다르면 채점이 어긋난다
    _a = fold(np.stack([to_target(r) for r in dev_rows]))[:, :N_CONF]
    _b = np.stack([to_multihot(r["labels"]) for r in dev_rows])[:, :N_CONF]
    assert (_a == _b).all(), f"🔴 dev 에서 접은 호 정답이 유형 정답과 다르다 — {int((_a != _b).any(1).sum())}행"

## 5. 손실함수 · 지표

🔴 v8.2 — `conf_metrics`(조기 종료 · 시드 선택 기준)는 **앞 6칸**으로만 잰다. 8종으로 바꾸지 않는다(팀장 10-06).

- Focal Loss(α = pos_weight). 원소(행×라벨)마다 `sample_weight × loss_mask`로 가중 평균을 낸다. 꺼진 원소는 분자와 분모 모두에서 빠진다.
- pos_weight는 마스크를 반영한 가중 개수로 계산한다. `POS_WEIGHT_SQRT`면 √(음성/양성)을 1~CAP으로 자른다.
- dev 지표는 **확정 6종만** 본다. 선택 기준은 **dev macro PR-AUC**(dev 양성이 있는 유형 평균)다.

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback, DataCollatorWithPadding, set_seed
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score, average_precision_score, f1_score

pos = (WM * Y).sum(axis=0)
neg = WM.sum(axis=0) - pos
ratio = neg / np.clip(pos, 1e-6, None)
pos_weight = np.clip(np.sqrt(ratio) if POS_WEIGHT_SQRT else ratio, 1, POS_WEIGHT_CAP)
print("pos_weight:", {l: round(float(pos_weight[i]), 2) for i, l in enumerate(TRAIN_LABELS)})
pos_weight_tensor = torch.tensor(pos_weight, dtype=torch.float32)

def elementwise_loss(logits, labels):
    pw = pos_weight_tensor.to(logits.device)
    logits = logits.float()
    if USE_FOCAL_LOSS:
        bce = torch.nn.functional.binary_cross_entropy_with_logits(logits, labels, reduction="none")
        p = torch.sigmoid(logits)
        p_t = p * labels + (1 - p) * (1 - labels)
        return (pw * labels + (1 - labels)) * (1 - p_t) ** FOCAL_GAMMA * bce
    return torch.nn.functional.binary_cross_entropy_with_logits(logits, labels, pos_weight=pw, reduction="none")

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        sw = inputs.pop("sample_weight", None)
        lm = inputs.pop("loss_mask", None)
        outputs = model(**inputs)
        el = elementwise_loss(outputs.logits, labels.float())
        wt = torch.ones_like(el)
        if sw is not None:
            wt = wt * sw.to(el.dtype)[:, None]
        if lm is not None:
            wt = wt * lm.to(el.dtype)
        loss = (el * wt).sum() / wt.sum().clamp_min(1e-6)
        return (loss, outputs) if return_outputs else loss

def safe(fn, y, p):
    return float(fn(y, p)) if 0 < y.sum() < len(y) else None

def conf_metrics(y, p):
    # 🔴 **기존 6종만** — y, p 는 (행, 전체 로짓). 조기 종료 · 가장 좋은 시드를 이 값으로 고른다. 8종으로 넓히지 않는다 —
    #    기준을 바꾸면 다른 회차 · 시드가 뽑혀 6종 수치가 2차 · 2회차와 이어지지 않는다 (팀장 10-06)
    out, prs, rocs = {}, [], []
    for i in range(N_CONF):
        pr, roc = safe(average_precision_score, y[:, i], p[:, i]), safe(roc_auc_score, y[:, i], p[:, i])
        out[f"prauc_{CONFIRMED[i]}"], out[f"rocauc_{CONFIRMED[i]}"] = pr, roc
        if pr is not None:
            prs.append(pr); rocs.append(roc)
    out["macro_prauc"] = float(np.mean(prs)) if prs else float("nan")
    out["macro_rocauc"] = float(np.mean(rocs)) if rocs else float("nan")
    _, _, f, _ = precision_recall_fscore_support(y[:, :N_CONF], (p[:, :N_CONF] > 0.5).astype(int), average="micro", zero_division=0)
    out["micro_f1_at05"] = f
    return out

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # 🆕 v8.1 — 호 판은 정답 · 확률을 유형으로 접은 뒤 잰다(유형 판에서는 fold 가 그대로 돌려준다) → 조기 종료 기준이 두 판에서 같다
    return conf_metrics(fold(labels).astype(int), fold(1 / (1 + np.exp(-logits))))

def pick_one_threshold(y, p):
    # dev에서 F1 최대 · 격자 0.1 ~ 0.9(0.025) · dev 양성 10건 미만이면 0.5 — v7 부터 같은 규칙이다
    if y.sum() < 10:
        return 0.5
    best_f, best_t = -1, 0.5
    for t in np.arange(0.1, 0.9001, 0.025):
        f = f1_score(y, p >= t, zero_division=0)
        if f > best_f:
            best_f, best_t = f, float(round(t, 3))
    return best_t

def pick_thresholds(y, p):
    # 🔄 v8.2 — **확정 8종**의 문턱을 같은 규칙으로 고른다(앞 6종의 값은 v8.1 과 같다). 편입 대기 칸은 꺼 둔다(TH_OFF)
    th = {CONF8[i]: pick_one_threshold(y[:, i], p[:, i]) for i in range(N8)}
    th[WAITING[0]] = TH_OFF
    return th

def swap_counts(y, pred):
    # 🆕 v8.2 — 비방 ↔ 부당 비교를 서로 바꿔 낸 수 (식품 6호 · 7호는 경계가 겹친다 · 팀장 10-06). y · pred 는 (행, 8종)
    #    정답이 한쪽뿐인 행만 센다. 「바꿔 냄」 = 정답 유형은 안 내고 상대 유형만 낸 것
    ia, ib = CONF8.index("부당_비교광고"), CONF8.index("비방광고")
    out = {}
    for gold, other in ((ib, ia), (ia, ib)):
        m = (y[:, gold] == 1) & (y[:, other] == 0)
        n = int(m.sum()); hit = int(pred[m, gold].sum())
        both = int(((pred[m, gold] == 1) & (pred[m, other] == 1)).sum())
        sw = int(((pred[m, gold] == 0) & (pred[m, other] == 1)).sum())
        out[CONF8[gold]] = {"n": n, "hit": hit, "both": both, "swapped": sw, "none": n - hit - sw}
    return out

def print_swap(sc):
    for l, d in sc.items():
        o = [x for x in NEW_CONF if x != l][0]
        if not d["n"]:
            print(f"  정답 {l:10s} (상대 유형 없이) 행 없음")
            continue
        print(f"  정답 {l:10s} n={d['n']:4d}  정답대로 냄 {d['hit']:4d} ({d['hit'] / d['n'] * 100:5.1f}% · 그중 {o} 도 같이 {d['both']}) · "
              f"🔁 {o} 로 바꿔 냄 {d['swapped']:3d} ({d['swapped'] / d['n'] * 100:4.1f}%) · 둘 다 안 냄 {d['none']:3d}  → 둘 중 하나라도 {(d['hit'] + d['swapped']) / d['n'] * 100:5.1f}%")

def predict_probs(mdl, rows, bs=64):
    mdl.eval()
    dev_ = next(mdl.parameters()).device
    out = []
    for s in range(0, len(rows), bs):
        enc = tokenizer([r["text"] for r in rows[s:s + bs]], truncation=True, max_length=128,
                        padding=True, return_tensors="pt").to(dev_)
        with torch.no_grad():
            out.append(torch.sigmoid(mdl(**enc).logits.float()).cpu().numpy())
    return fold(np.concatenate(out)) if out else np.zeros((0, len(LABEL_LIST)))   # 🆕 v8.1 — 언제나 유형 공간으로 돌려준다

## 6. 학습 — 시드마다 한 번 (조기 종료)

🆕 v8.2 — 같은 세션에 **같은 설정으로 학습한 시드가 있으면 다시 학습하지 않는다**(`♻️`). `FINAL_TEST` 만 바꿔 다시 돌릴 때 dev 를 본 그 모델을 그대로 test 에 올린다.


시드마다 dev macro PR-AUC가 가장 높은 epoch의 가중치를 `./runs/<실험>/seed<시드>`에 저장하고, 그 dev에서 threshold를 고른다.

🆕 v9 — 문턱은 `THRESHOLD_MODE`(기본 `robust`)를 **적용 문턱**으로 쓴다. dev 조건 지표 · 저장 · test 가 모두 이 값을 쓴다. 셀 끝에 **두 규칙을 dev 로 견준 표**가 붙는다.


In [ ]:
slug = "".join(ch if ch.isalnum() else "_" for ch in exp_key)[:80]
RUN_DIR = f"./runs/{slug}"
collator = DataCollatorWithPadding(tokenizer)
dev_y = np.stack([to_multihot(r["labels"]) for r in dev_rows]).astype(int)
# 🆕 v8.2 — 같은 세션 · 같은 설정 · 같은 학습 행이면 학습한 시드를 다시 쓴다. FINAL_TEST 만 바꿔 다시 돌릴 때(B 회차)
#    dev 를 본 **바로 그 모델**을 test 에 올리려는 것이다 — 다시 학습하면 GPU 연산 순서 때문에 가중치가 조금 달라질 수 있다
_train_sig = (exp_key, GOLDEN_SHA, tuple(SEEDS), MAX_EPOCHS, PATIENCE, BATCH, LR, WARMUP,
              hashlib.sha256("|".join(r["id"] for r in train_rows).encode()).hexdigest(),
              hashlib.sha256("|".join(r["id"] for r in dev_rows).encode()).hexdigest())
REUSE = (globals().get("_TRAINED_SIG") == _train_sig and set(globals().get("seed_results", {})) == set(SEEDS)
         and all(os.path.exists(os.path.join(seed_results[s]["dir"], "config.json")) for s in SEEDS))
if REUSE:
    print(f"♻️ 이 세션에서 같은 설정으로 학습한 시드 {SEEDS} 를 그대로 쓴다 — 다시 학습하지 않는다 (dev 를 본 그 모델)")
else:
    seed_results = {}

for seed in ([] if REUSE else SEEDS):
    print(f"\n==================== seed {seed} ====================")
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=len(TRAIN_LABELS), problem_type="multi_label_classification",
        id2label=dict(enumerate(TRAIN_LABELS)), label2id=TRAIN2ID)
    args = TrainingArguments(
        output_dir=f"{RUN_DIR}/ckpt{seed}", eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
        per_device_train_batch_size=BATCH, per_device_eval_batch_size=64, num_train_epochs=MAX_EPOCHS,
        learning_rate=LR, weight_decay=0.01, lr_scheduler_type="linear",
        warmup_steps=int(WARMUP * math.ceil(len(train_ds) / BATCH) * MAX_EPOCHS),
        load_best_model_at_end=True, metric_for_best_model="macro_prauc", greater_is_better=True,
        logging_steps=50, report_to="none", remove_unused_columns=False, seed=seed,
        fp16=torch.cuda.is_available())
    trainer = WeightedTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=dev_ds,
                              data_collator=collator, compute_metrics=compute_metrics,
                              callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)])
    trainer.train()

    hist = [h for h in trainer.state.log_history if "eval_macro_prauc" in h]
    best = max(hist, key=lambda h: h["eval_macro_prauc"])
    dev_p = predict_probs(trainer.model, dev_rows)
    dm = conf_metrics(dev_y, dev_p)
    th = pick_thresholds(dev_y, dev_p)
    save_dir = f"{RUN_DIR}/seed{seed}"
    trainer.save_model(save_dir)
    shutil.rmtree(f"{RUN_DIR}/ckpt{seed}", ignore_errors=True)
    seed_results[seed] = {"dev": dm, "best_epoch": int(round(best["epoch"])), "epochs_run": int(round(hist[-1]["epoch"])),
                          "thresholds": th, "dir": save_dir, "dev_p": dev_p, "dev_u_p": predict_probs(trainer.model, DEV_UNTYPED),
                          "dev_inj_p": predict_probs(trainer.model, DEV_INJ),
                          "curve": [(int(round(h["epoch"])), round(h["eval_macro_prauc"], 4)) for h in hist]}
    print(f"seed {seed}: dev macro PR-AUC {dm['macro_prauc']:.3f} (best epoch {seed_results[seed]['best_epoch']}) · 곡선 {seed_results[seed]['curve']}")
    del trainer, model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

_TRAINED_SIG = _train_sig
devs = [v["dev"]["macro_prauc"] for v in seed_results.values()]
BEST_SEED = max(seed_results, key=lambda s: seed_results[s]["dev"]["macro_prauc"])
print(f"\n[{exp_key}] dev macro PR-AUC {np.mean(devs):.3f} ± {np.std(devs):.3f}  (시드 {SEEDS}) · 가장 좋은 시드 {BEST_SEED}")
print(f"{'유형':20s} " + " ".join(f"{'s'+str(s):>8s}" for s in SEEDS))
for l in CONFIRMED:
    vals = [seed_results[s]["dev"].get(f"prauc_{l}") for s in SEEDS]
    print(f"{l:20s} " + " ".join(f"{v:8.3f}" if v is not None else "     N/A" for v in vals))

# ── v7.2 robust threshold: 시드마다의 dev 예측에서 F1을 구해 평균이 가장 높은 threshold
def pick_thresholds_robust(y, probs_list):
    th = {}
    for i in range(N8):   # 🔄 v8.2 — 확정 8종
        if y[:, i].sum() < 10:
            th[CONF8[i]] = 0.5
            continue
        grid = np.arange(0.1, 0.9001, 0.025)
        mean_f = [np.mean([f1_score(y[:, i], p[:, i] >= t, zero_division=0) for p in probs_list]) for t in grid]
        th[CONF8[i]] = float(round(grid[int(np.argmax(mean_f))], 3))
    th[WAITING[0]] = TH_OFF
    return th

TH_ROBUST = pick_thresholds_robust(dev_y, [seed_results[s]["dev_p"] for s in SEEDS])
print(f"\nthreshold (THRESHOLD_MODE={THRESHOLD_MODE})")
print(f"{'유형':20s} " + " ".join(f"{'s'+str(s):>7s}" for s in SEEDS) + f" {'robust':>8s}")
for l in CONF8:
    print(f"{l:20s} " + " ".join(f"{seed_results[s]['thresholds'][l]:7.3f}" for s in SEEDS) + f" {TH_ROBUST[l]:8.3f}" + ("   🆕 확정 (D-321)" if l in NEW_CONF else ""))
print(f"{WAITING[0]:20s} " + " ".join(f"{TH_OFF:7.3f}" for s in SEEDS) + f" {TH_OFF:8.3f}   편입 대기 — 꺼 둠")

# ── dev 조건 지표 (가장 좋은 시드 · 그 시드의 dev threshold) — 실험 비교는 이것으로 (D-175)
def wilson(k, n, z=1.96):
    """Wilson 95% 신뢰구간 (D-40)."""
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (max(0.0, c - h), min(1.0, c + h))

# 🆕 v9 — dev 지표 · 저장 문턱 · test 가 모두 **적용 문턱**(THRESHOLD_MODE)을 쓴다. v8.3 까지는 dev 지표를 늘 best_seed 문턱으로 쟀다
TH_APPLIED = TH_ROBUST if THRESHOLD_MODE == "robust" else seed_results[BEST_SEED]["thresholds"]
_th_dev = np.array([TH_APPLIED[l] for l in CONFIRMED])
_dp = seed_results[BEST_SEED]["dev_p"][:, :N_CONF]
_dy = dev_y[:, :N_CONF]
_pos = _dy.any(1)
_flag = (_dp >= _th_dev).any(1)
# 🔄 v8 — dev 음성은 **유형이 하나도 없는 행**만이다. 보조 유형(비방 · 부당비교 · 기능성화장품)만 있는 행은 위반이라 음성에서 뺀다
#    (v7.x 는 이 행들을 음성으로 세었다 — 위반 문구에 울린 것이 「음성 오판정」으로 잡혔다)
_neg = np.array([not r["labels"] for r in dev_rows], dtype=bool)
dev_cond = {"dev_violation_recall": float(_flag[_pos].mean()) if _pos.any() else float("nan"),
            "dev_neg_fpr": float(_flag[_neg].mean()) if _neg.any() else float("nan"),
            "n_pos": int(_pos.sum()), "n_neg": int(_neg.sum()), "n_aux_only": int((~_pos & ~_neg).sum()),
            "k_pos_hit": int(_flag[_pos].sum()), "k_neg_flag": int(_flag[_neg].sum())}  # 줄 수(사전 고정 기준 G1 · G3)
_dc = np.array([r["condition"] or "없음" for r in dev_rows])
for _c in sorted(set(_dc[_neg])):   # 🆕 v8 — dev 음성 오판정을 조건별로 (D · L · 없음)
    _m = _neg & (_dc == _c)
    dev_cond[f"n_neg:{_c}"] = int(_m.sum()); dev_cond[f"k_neg_flag:{_c}"] = int(_flag[_m].sum())
_up = seed_results[BEST_SEED]["dev_u_p"]   # 🆕 v8.1 — dev 묶음의 유형 없는 위반(학습 안 함)을 얼마나 잡나 · 참고
if len(_up):
    _uflag = (_up[:, :N_CONF] >= _th_dev).any(1)
    if LABEL_UNIT == "ho":      # 유형 없는 호의 로짓 — 문턱을 고를 dev 양성이 없어 0.5 로 둔다
        _uflag = _uflag | (_up[:, LABEL2ID[UNTYPED]] >= 0.5)
    dev_cond["n_untyped"] = int(len(_up)); dev_cond["k_untyped_hit"] = int(_uflag.sum())
if extra_holdout:
    _best_tmp = AutoModelForSequenceClassification.from_pretrained(seed_results[BEST_SEED]["dir"])
    if torch.cuda.is_available():
        _best_tmp = _best_tmp.cuda()
    for _f, _rows in extra_holdout.items():
        if not _rows:
            continue
        _hp = predict_probs(_best_tmp, _rows)[:, :N_CONF] >= _th_dev
        _stem = os.path.splitext(os.path.basename(_f))[0]
        if any(r["labels"] for r in _rows):   # 라벨 있는 파일 — 정답 유형 탐지율 (높을수록 좋다)
            _y = np.stack([to_multihot(r["labels"]) for r in _rows])[:, :N_CONF].astype(bool)
            dev_cond[f"hold:{_stem}:탐지율"] = float((_hp & _y).any(1).mean())
            dev_cond[f"k_hold:{_stem}"] = int((_hp & _y).any(1).sum())
        else:                                  # 음성 파일 — 6종 중 하나라도 울린 비율 (낮을수록 좋다)
            dev_cond[f"hold:{_stem}:오탐률"] = float(_hp.any(1).mean())
        dev_cond[f"n_hold:{_stem}"] = len(_rows)
    del _best_tmp
print(f"\n[dev 조건 지표 · 시드 {BEST_SEED} · 문턱 {THRESHOLD_MODE}]")
for k, v in dev_cond.items():
    print(f"  {k:22s} {v*100:6.1f}%" if isinstance(v, float) else f"  {k:22s} {v}")
print("\n[dev 유형별 P · R (Wilson 95% CI · D-40)]")
for i, l in enumerate(CONFIRMED):
    n = int(_dy[:, i].sum()); tp = int(((_dp[:, i] >= _th_dev[i]) & (_dy[:, i] == 1)).sum()); pp = int((_dp[:, i] >= _th_dev[i]).sum())
    rl, rh = wilson(tp, n); pl, ph = wilson(tp, pp)
    dev_cond[f"R:{l}"] = tp / n if n else float("nan"); dev_cond[f"n_R:{l}"] = n; dev_cond[f"k_R:{l}"] = tp  # v7.5 — G2 · 유형별 표
    print(f"  {l:20s} n={n:4d}  R {tp/max(n,1):.3f} [{rl:.3f}, {rh:.3f}]  P {tp/max(pp,1):.3f} [{pl:.3f}, {ph:.3f}]" + ("  측정 불가(n<30)" if n < 30 else ""))


# ══ 🆕 v8.2 (D-321) dev 8종 — 부당 비교 · 비방의 문턱도 dev 에서 골랐다(위 표 · 같은 규칙). 위의 6종 지표 · 모델 선택은 그대로다
_th8 = np.array([TH_APPLIED[l] for l in CONF8])
_dpa = seed_results[BEST_SEED]["dev_p"]
_dp8, _dy8 = _dpa[:, :N8], dev_y[:, :N8]
_pos8, _flag8 = _dy8.any(1), (_dp8 >= _th8).any(1)
_real = np.array([not is_inj(r) for r in dev_rows], dtype=bool)
dev8 = {"n_pos8": int(_pos8.sum()), "k_pos_hit8": int(_flag8[_pos8].sum()),
        "dev_violation_recall8": float(_flag8[_pos8].mean()) if _pos8.any() else float("nan"),
        "n_neg": int(_neg.sum()), "k_neg_flag8": int(_flag8[_neg].sum()), "n_wait_only": int((~_pos8 & ~_neg).sum()),
        "n_dev": len(dev_rows), "n_dev_real": int(_real.sum())}
print(f"\n[dev 6종 기준 → 8종 기준 · 시드 {BEST_SEED} · 같은 모델 · 같은 dev 행]")
print(f"  위반 재현율   {dev_cond['k_pos_hit']:4d}/{dev_cond['n_pos']:<4d} {dev_cond['dev_violation_recall'] * 100:5.1f}%  →  "
      f"{dev8['k_pos_hit8']:4d}/{dev8['n_pos8']:<4d} {dev8['dev_violation_recall8'] * 100:5.1f}%   (양성이 {dev8['n_pos8'] - dev_cond['n_pos']}줄 는다 — 부당 비교 · 비방만 있는 행)")
print(f"  음성 오판정   {dev_cond['k_neg_flag']:4d}/{dev_cond['n_neg']:<4d} {dev_cond['dev_neg_fpr'] * 100:5.1f}%  →  "
      f"{dev8['k_neg_flag8']:4d}/{dev8['n_neg']:<4d} {dev8['k_neg_flag8'] / max(dev8['n_neg'], 1) * 100:5.1f}%   (새 2종이 더한 오판정 {dev8['k_neg_flag8'] - dev_cond['k_neg_flag']:+d}줄)")
for _c in sorted(set(_dc[_neg])):
    _m = _neg & (_dc == _c)
    dev8[f"k_neg_flag8:{_c}"] = int(_flag8[_m].sum())
    print(f"    조건 {_c:3s}     {dev_cond[f'k_neg_flag:{_c}']:4d}/{int(_m.sum()):<4d}        →  {dev8[f'k_neg_flag8:{_c}']:4d}/{int(_m.sum()):<4d}")
print(f"  (편입 대기 유형만 있는 dev 행 {dev8['n_wait_only']}줄은 양성도 음성도 아니다)")

print("\n[dev 새 확정 2종 — " + ("실제 문구만 (dev 에 주입 행이 없다)" if _real.all() else "전체 / 주입(합성) 제외") + " · Wilson 95% CI · PR-AUC 는 시드 평균]")
for l in NEW_CONF:
    i, t = LABEL2ID[l], float(_th8[LABEL2ID[l]])
    for name, m in ([("전체", np.ones(len(dev_rows), dtype=bool))] + ([] if _real.all() else [("주입 제외", _real)])):
        y_, p_ = dev_y[m, i], _dpa[m, i]
        n = int(y_.sum()); tp = int(((p_ >= t) & (y_ == 1)).sum()); pp = int((p_ >= t).sum())
        prs = [safe(average_precision_score, dev_y[m, i], seed_results[s]["dev_p"][m, i]) for s in SEEDS]
        prs = [v for v in prs if v is not None]
        rl, rh = wilson(tp, n); pl, ph = wilson(tp, pp)
        dev8[f"{l}|{name}"] = {"rows": int(m.sum()), "n": n, "k": tp, "pp": pp, "prauc_mean": float(np.mean(prs)) if prs else None}
        print(f"  {l:10s} {name:6s} 행 {int(m.sum()):4d} · 양성 n={n:3d}  R {tp / max(n, 1):.3f} [{rl:.3f}, {rh:.3f}]  P {tp / max(pp, 1):.3f} [{pl:.3f}, {ph:.3f}]"
              f"  PR-AUC {np.mean(prs):.3f}" if prs else f"  {l:10s} {name:6s} 행 {int(m.sum()):4d} · 양성 n={n:3d}  PR-AUC N/A", end="")
        print("  측정 불가(n<30)" if n < 30 else "")
    _t_real = pick_one_threshold(dev_y[_real, i], _dpa[_real, i])
    dev8[f"th:{l}"] = t; dev8[f"th_real_only:{l}"] = _t_real
    _a, _b = DEV_NEW_N[l]
    print((f"    문턱 {t:.3f} (실제 문구 dev 로 골랐다 — 적용값) · dev 양성 {_a}" if _real.all() else
           f"    문턱 {t:.3f} (dev 전체로 골랐다 — 적용값) · 참고: 주입 제외 행으로만 골랐다면 {_t_real:.3f} · dev 양성 {_a} 중 주입 {_a - _b}")
          + ("  🔴 양성 10건 미만 — 0.5 고정" if _a < 10 else ""))
_n2 = [dev8[f"{l}|전체"]["prauc_mean"] for l in NEW_CONF if dev8[f"{l}|전체"]["prauc_mean"] is not None]
dev8["new2_prauc_mean"] = float(np.mean(_n2)) if _n2 else None    # 🆕 v8.3 — 채택 조건(섹션 9)에 쓴다
print(f"  새 2종 dev PR-AUC 평균 {'N/A' if dev8['new2_prauc_mean'] is None else round(dev8['new2_prauc_mean'], 3)}")
_ip = seed_results[BEST_SEED].get("dev_inj_p")
if _ip is not None and len(_ip):   # 🆕 v8.3 — 채점에서 뺀 주입 행 (참고)
    _iy = np.stack([to_multihot(r["labels"]) for r in DEV_INJ])[:, :N8].astype(int)
    dev8["inj_ref"] = {l: [int(((_ip[:, i] >= _th8[i]) & (_iy[:, i] == 1)).sum()), int(_iy[:, i].sum())] for i, l in enumerate(CONF8)}
    print("\n[참고 — dev 묶음의 주입(합성) 행 · 채점 밖 · 학습 안 함] 유형별 탐지:",
          {l: f"{k}/{n}" for l, (k, n) in dev8["inj_ref"].items() if n})
print("\n[dev 비방 ↔ 부당 비교를 바꿔 낸 수 — 정답이 한쪽뿐인 행]")
dev8["swap"] = swap_counts(_dy8, (_dp8 >= _th8).astype(int))
print_swap(dev8["swap"])
_yw, _pw = dev_y[:, IDX_WAIT], _dpa[:, IDX_WAIT]
dev8["wait"] = {"n": int(_yw.sum()), "k_at05": int(((_pw >= WAIT_REF_TH) & (_yw == 1)).sum()), "pp_at05": int((_pw >= WAIT_REF_TH).sum())}
print(f"\n[dev 편입 대기 · {WAITING[0]}] 양성 n={dev8['wait']['n']} · {WAIT_REF_TH} 기준으로 잡은 것 {dev8['wait']['k_at05']} · 울린 것 {dev8['wait']['pp_at05']}  (참고 — 문턱은 꺼 둔다)")


# ══ 🆕 v9 — 문턱 규칙 둘을 dev 로 견준다 (같은 모델 = 시드 BEST_SEED · 같은 dev 행). test 는 쓰지 않는다
def _dev_by_rule(thd):
    t6, t8 = np.array([thd[l] for l in CONFIRMED]), np.array([thd[l] for l in CONF8])
    f6, f8 = _dp >= t6, _dp8 >= t8
    out = {"k_pos_hit": int(f6.any(1)[_pos].sum()), "k_neg_flag": int(f6.any(1)[_neg].sum()),
           "k_pos_hit8": int(f8.any(1)[_pos8].sum()), "k_neg_flag8": int(f8.any(1)[_neg].sum()), "label": {}}
    for i, l in enumerate(CONF8):
        y = dev_y[:, i] == 1
        tp, pp = int((f8[:, i] & y).sum()), int(f8[:, i].sum())
        out["label"][l] = [tp, int(y.sum()), pp]
    return out
DEV_RULES = {"best_seed": _dev_by_rule(seed_results[BEST_SEED]["thresholds"]), "robust": _dev_by_rule(TH_ROBUST)}
print(f"\n[🆕 v9 문턱 규칙 둘 — dev 에서의 차이 · 시드 {BEST_SEED} · 적용 = {THRESHOLD_MODE}]")
print(f"{'':22s} {'best_seed':>22s} {'robust':>22s}")
for name, k, n in (("위반 재현율 (6종)", "k_pos_hit", int(_pos.sum())), ("음성 오판정 (6종)", "k_neg_flag", int(_neg.sum())),
                   ("위반 재현율 (8종)", "k_pos_hit8", int(_pos8.sum())), ("음성 오판정 (8종)", "k_neg_flag8", int(_neg.sum()))):
    print(f"{name:22s} " + " ".join(f"{DEV_RULES[r][k]:8d}/{n:<5d} {DEV_RULES[r][k] / max(n, 1) * 100:5.1f}%" for r in ("best_seed", "robust")))
print(f"{'유형 (문턱 · 재현율 · 정밀도)':22s}")
for l in CONF8:
    cells = []
    for r, thd in (("best_seed", seed_results[BEST_SEED]["thresholds"]), ("robust", TH_ROBUST)):
        tp, n, pp = DEV_RULES[r]["label"][l]
        cells.append(f"@{thd[l]:.3f} {tp:3d}/{n:<3d} P {tp / max(pp, 1):.2f}")
    print(f"  {l:20s} " + "   ".join(f"{c:>20s}" for c in cells) + ("  측정 불가(n<30)" if DEV_RULES["robust"]["label"][l][1] < 30 else ""))
print("  · 문턱이 올라간 유형은 재현율이 내려가고 정밀도가 오른다. 이 표가 v9 에서 문턱 규칙을 바꾼 효과의 전부다(모델은 같다)")


## 7. test 채점 (후보당 한 번 · 보고용) — 가장 좋은 시드 · dev threshold

후보 선택은 섹션 6의 dev 평균으로 한다. 여기 수치는 선택에 쓰지 않는다. 🆕 v8.2 — **6종 기준 표와 8종 기준 표를 둘 다** 낸다(같은 모델 · 같은 문턱).

| 표 | 채점 행 | 플래그 |
|---|---|---|
| **6종 기준** (v8 까지와 같은 정의 — 2차와 이어 보는 표) | `scored` — 기존 6종만 정답인 행 | 6종 중 하나라도 문턱 이상 |
| **8종 기준** (D-321) | `scored8` — 8종 중 하나라도 정답인 행 | 8종 중 하나라도 문턱 이상 |

- 🔄 **유형 없는 위반(평가 131 · 화장품법 13①4 등)** 은 양성 재현율의 분모에서 빼고 **따로** 센다 — 인코더가 낼 수 없는 답이다(팀장 10-06).
- **편입 대기**(`기능성화장품_오인`)는 0.5 기준 참고값으로 따로 낸다 — 앱에서는 문턱이 꺼져 있어 이 유형을 내지 않는다.
- 비방 ↔ 부당 비교를 **바꿔 낸 수** · **거짓_과장 따로 보기**(호별 · 조건별)가 붙는다.
- 🆕 v9 — 끝에 **dev → test** · **호별 · 법별 8종 기준** · **행별 확률 csv**(문구 없음)가 붙는다. 「threshold 방식 비교」 줄은 두 규칙을 같은 모델로 test 에서 나란히 찍는다 — 적용은 `←` 표시가 붙은 쪽이고 test 를 보고 바꾸지 않는다.


In [ ]:
import csv
best_model = AutoModelForSequenceClassification.from_pretrained(seed_results[BEST_SEED]["dir"])
if torch.cuda.is_available():
    best_model = best_model.cuda()
TH_BEST = seed_results[BEST_SEED]["thresholds"]
TH = TH_ROBUST if THRESHOLD_MODE == "robust" else TH_BEST
print(f"적용 threshold: {THRESHOLD_MODE} · 모델: 시드 {BEST_SEED}")

# ── v7.4: 🔴 test 는 FINAL_TEST 일 때만 채점한다 (D-175)
micro_dev = micro_05 = macro_meas = f5ho = None
test8 = {}   # 🆕 v8.2 — 8종 기준 표 · 바꿔 낸 수 · 편입 대기 참고값 · 거짓_과장
tm, test_label, bucket_summary, law_f1, ho_hit, gate_rows = {}, {}, {}, {}, {}, []
if not FINAL_TEST:
    print("🔒 test 봉인 — FINAL_TEST=False. 실험 비교는 섹션 6 끝의 dev 조건 지표로 한다 (D-175)")
else:
    print("🔴 최종 측정 — 이 결과로 설정 · 데이터를 다시 고르지 않는다 (D-175)")
    th_vec = np.array([TH[l] for l in CONFIRMED])

    scored_rows = BUCKETS["scored"]
    y_te = np.stack([to_multihot(r["labels"]) for r in scored_rows])[:, :N_CONF].astype(int)
    p_te_all = predict_probs(best_model, scored_rows)
    p_te = p_te_all[:, :N_CONF]
    pred_dev = (p_te >= th_vec).astype(int)
    pred_05 = (p_te > 0.5).astype(int)
    tm = conf_metrics(np.hstack([y_te, np.zeros((len(y_te), len(LABEL_LIST) - N_CONF), int)]), p_te_all)

    print(f"[test **6종 기준** · scored {len(scored_rows)}행 · 시드 {BEST_SEED}] — v8 까지와 같은 정의 (2차와 이어 보는 표)")
    print(f"{'유형':20s} {'정답':>5s} {'th':>6s} {'P':>6s} {'R':>6s} {'F1':>6s} {'F1@0.5':>7s} {'ROC':>6s} {'PR-AUC':>7s}")
    test_label = {}
    for i, l in enumerate(CONFIRMED):
        n = int(y_te[:, i].sum())
        p_, r_, f_, _ = precision_recall_fscore_support(y_te[:, i], pred_dev[:, i], average="binary", zero_division=0)
        f05 = f1_score(y_te[:, i], pred_05[:, i], zero_division=0)
        roc, pr = tm.get(f"rocauc_{l}"), tm.get(f"prauc_{l}")
        note = "  측정 불가(n<30)" if n < 30 else ""
        test_label[l] = {"n": n, "th": TH[l], "P": p_, "R": r_, "F1": f_, "F1@0.5": f05, "rocauc": roc, "prauc": pr}
        fmt = lambda v: f"{v:6.3f}" if v is not None else "   N/A"
        tp_ = int((pred_dev[:, i] & y_te[:, i]).sum()); pp_ = int(pred_dev[:, i].sum())
        rci, pci = wilson(tp_, n), wilson(tp_, pp_)
        test_label[l].update({"R_ci": rci, "P_ci": pci})
        print(f"{l:20s} {n:5d} {TH[l]:6.3f} {p_:6.3f} {r_:6.3f} {f_:6.3f} {f05:7.3f} {fmt(roc)} {fmt(pr):>7s}{note}  R CI [{rci[0]:.2f},{rci[1]:.2f}] P CI [{pci[0]:.2f},{pci[1]:.2f}]")

    _, _, micro_dev, _ = precision_recall_fscore_support(y_te, pred_dev, average="micro", zero_division=0)
    _, _, micro_05, _ = precision_recall_fscore_support(y_te, pred_05, average="micro", zero_division=0)
    gi = [CONFIRMED.index("소비자_기만"), CONFIRMED.index("후기_체험기_기만")]
    f5ho = f1_score(y_te[:, gi].any(1), pred_dev[:, gi].any(1), zero_division=0)
    measurable = [l for l in CONFIRMED if test_label[l]["n"] >= 30]
    macro_meas = float(np.mean([test_label[l]["F1"] for l in measurable]))
    print(f"\nmicro F1 {micro_dev:.3f} (0.5 고정 {micro_05:.3f}) · 측정 가능 {len(measurable)}종 macro F1 {macro_meas:.3f} · "
          f"5호 묶음 F1 {f5ho:.3f} · macro PR-AUC {tm['macro_prauc']:.3f} · macro ROC-AUC {tm['macro_rocauc']:.3f}")

    # ── v7.2 참고: 두 threshold 방식 비교 (같은 모델 · 보고용)
    def th_summary(thd):
        tv = np.array([thd[l] for l in CONFIRMED])
        pr = (p_te >= tv).astype(int)
        _, _, mf, _ = precision_recall_fscore_support(y_te, pr, average="micro", zero_division=0)
        mac = np.mean([f1_score(y_te[:, CONFIRMED.index(l)], pr[:, CONFIRMED.index(l)], zero_division=0) for l in measurable])
        neg = float((predict_probs(best_model, BUCKETS["neg"])[:, :N_CONF] >= tv).any(1).mean())
        return mf, mac, float(pr.any(1).mean()), neg
    print("\n[threshold 방식 비교 · 같은 모델]  micro F1 / 측정가능 macro F1 / 위반 탐지 재현율 / 음성 오판정")
    for name, thd in [("best_seed", TH_BEST), ("robust", TH_ROBUST)]:
        mf, mac, vr, ng = th_summary(thd)
        print(f"  {name:10s} {mf:.3f} / {mac:.3f} / {vr*100:.1f}% / {ng*100:.1f}%" + ("   ← 적용" if thd is TH else ""))

    # ── 조건 버킷
    bucket_probs = {b: predict_probs(best_model, BUCKETS[b])[:, :N_CONF] for b in ["neg", "D", "M", "pending", "aux"]}
    flag = lambda p: float((p >= th_vec).any(axis=1).mean()) if len(p) else float("nan")
    bucket_summary = {"neg_fpr": flag(bucket_probs["neg"]), "D_flag": flag(bucket_probs["D"]),
                      "M_flag": flag(bucket_probs["M"]), "pending_recall": flag(bucket_probs["pending"]),
                      "violation_recall": float((pred_dev.any(axis=1)).mean())}
    print("\n=== 조건 버킷 (dev threshold · 확정 6종 중 하나라도) ===")
    for k, b in [("neg_fpr", "neg"), ("D_flag", "D"), ("M_flag", "M"), ("pending_recall", "pending")]:
        print(f"  {k:16s} {bucket_summary[k]*100:6.1f}%  (n={len(BUCKETS[b])})")
    print(f"  위반 탐지 재현율  {bucket_summary['violation_recall']*100:6.1f}%  (scored {len(scored_rows)})")
    print(f"  aux(확정 6종 밖 위반) 플래그율 {flag(bucket_probs['aux'])*100:.1f}%  (n={len(BUCKETS['aux'])} · 참고)")

    # ── 🆕 10-01 팀장 채점 기준 — C·A·B 양성 · L 음성 · D 별도 지표 · M 채점 안 함
    # 🔄 v8.2 (팀장 10-06) — 유형 없는 위반(pending · 131)은 인코더가 낼 수 없는 답이다 → 양성 재현율의 분모에서 빼고 따로 센다
    _un_k = int((bucket_probs["pending"] >= th_vec).any(axis=1).sum()); _un_n = len(BUCKETS["pending"])
    _pos_k = int(pred_dev.any(axis=1).sum())
    _pos_n = len(scored_rows)
    _neg_k = int((bucket_probs["neg"] >= th_vec).any(axis=1).sum()); _neg_n = len(BUCKETS["neg"])
    _d_k = int((bucket_probs["D"] >= th_vec).any(axis=1).sum()); _d_n = len(BUCKETS["D"])
    bucket_summary.update({"team_pos_recall": _pos_k / _pos_n if _pos_n else float("nan"),
                           "team_L_fpr": _neg_k / _neg_n if _neg_n else float("nan"),
                           "team_D_flag": _d_k / _d_n if _d_n else float("nan"),
                           "team_n": {"pos": _pos_n, "L": _neg_n, "D": _d_n, "M": len(BUCKETS["M"])}})
    bucket_summary.update({"untyped_flag": [_un_k, _un_n],
                           "team_pos_recall_with_untyped": (_pos_k + _un_k) / (_pos_n + _un_n) if _pos_n + _un_n else float("nan")})
    print("\n=== 팀장 채점 기준 · 6종 기준 (유형 있는 위반 양성 · L 음성 · D 별도 · M · 유형 없는 위반은 채점 안 함) ===")
    for name, k, n, note in [("양성 탐지 재현율 (유형 있는 위반)", _pos_k, _pos_n, "높을수록 좋다"),
                             ("유형 없는 위반 탐지 (따로)", _un_k, _un_n, "채점 밖 — 인코더가 낼 수 없는 답(화장품법 13①4 등). 다른 유형으로 울린 수다"),
                             ("음성 오판정률 (L)", _neg_k, _neg_n, "낮을수록 좋다"),
                             ("D 플래그율 (별도 지표)", _d_k, _d_n, "낮을수록 좋다 — 인코더는 not_claim 을 내지 않는다(D-275)")]:
        if n:
            lo_, hi_ = wilson(k, n)
            print(f"  {name:22s} {k:5d}/{n:<5d} {k / n * 100:5.1f}% [{lo_ * 100:.1f}~{hi_ * 100:.1f}%]  {note}{'  측정 불가(n<30)' if n < 30 else ''}")
    print(f"  M (채점 안 함) n={len(BUCKETS['M'])} · 위 M_flag 는 참고")
    if _pos_n + _un_n:
        print(f"  참고 — 유형 없는 위반까지 분모에 넣으면(v8.1 까지의 정의) {(_pos_k + _un_k) / (_pos_n + _un_n) * 100:.1f}% ({_pos_k + _un_k}/{_pos_n + _un_n})")
    _aux_k = int((bucket_probs["aux"] >= th_vec).any(axis=1).sum()); _aux_n = len(BUCKETS["aux"])
    if _pos_n + _aux_n:
        print(f"  참고 — 확정 6종 밖 위반(aux)까지 넣은 탐지율 {(_pos_k + _aux_k) / (_pos_n + _aux_n) * 100:.1f}% ({_pos_k + _aux_k}/{_pos_n + _aux_n})")

    if LABEL_UNIT == "ho" and len(BUCKETS["pending"]):   # 🆕 v8.1 — 호 판만: 유형 없는 호 로짓(0.5)까지 넣으면 pending 을 얼마나 잡나
        _pu = predict_probs(best_model, BUCKETS["pending"])
        _pk = int(((_pu[:, :N_CONF] >= th_vec).any(axis=1) | (_pu[:, LABEL2ID[UNTYPED]] >= 0.5)).sum())
        bucket_summary["pending_hit_with_untyped"] = [_pk, len(BUCKETS["pending"])]
        print(f"  참고 — 유형 없는 호 로짓까지 넣은 pending 탐지 {_pk}/{len(BUCKETS['pending'])} {_pk / len(BUCKETS['pending']) * 100:.1f}%")

    # ── 🆕 v8 (D-301) 적법 문장 오탐 — 내역과 함께만 낸다
    _dflag = (bucket_probs["D"] >= th_vec).any(axis=1)
    _gf = np.array([r["id"].startswith("gf:") for r in BUCKETS["D"]], dtype=bool)
    _nc_k, _nc_n = int(_dflag[_gf].sum()), int(_gf.sum())
    _gd_k, _gd_n = int(_dflag[~_gf].sum()), int((~_gf).sum())
    bucket_summary.update({"lawful_claim": [_neg_k, _neg_n], "lawful_noclaim": [_nc_k, _nc_n], "D_guide_flag": [_gd_k, _gd_n],
                           "aux_flag": [_aux_k, _aux_n]})
    print("\n=== 적법 문장 오탐 (D-301 · 내역과 함께) ===")
    for name, k, n in [("주장 있음 (조건 L)", _neg_k, _neg_n), ("주장 없음 (수정문구 판 D)", _nc_k, _nc_n)]:
        if n:
            lo_, hi_ = wilson(k, n)
            print(f"  {name:24s} {k:5d}/{n:<5d} {k / n * 100:5.1f}% [{lo_ * 100:.1f}~{hi_ * 100:.1f}%]{'  측정 불가(n<30)' if n < 30 else ''}")
        else:
            print(f"  {name:24s} 행 없음")
    if _gd_n:
        print(f"  참고 — 해설서 위반문구 D {_gd_k}/{_gd_n} {_gd_k / _gd_n * 100:.1f}%  (적법 문장이 아니다 — 기관이 삭제를 지시한 문구 · 오탐률로 세지 않는다)")

    # ── 법별 · 조건별 · 호별
    hit = (pred_dev & y_te).any(axis=1)
    law_f1 = {}
    print("\n=== 법별 micro F1 ===")
    for law in sorted({lw for r in scored_rows for lw in r["laws"]}):
        idx = [k for k, r in enumerate(scored_rows) if law in r["laws"]]
        _, _, f, _ = precision_recall_fscore_support(y_te[idx], pred_dev[idx], average="micro", zero_division=0)
        law_f1[law] = f
        print(f"  {LAW_NAMES.get(law, law):12s} n={len(idx):5d}  micro F1={f:.3f}")
    print("\n=== 조건별 적중률 (정답 유형 중 하나라도) ===")
    for c, name in [("C", "C 표현 자체"), ("A", "A 조성·지위"), ("B", "B 실증 필요"), (None, "조건 없음")]:
        idx = [k for k, r in enumerate(scored_rows) if r["condition"] == c]
        if idx:
            print(f"  {name:16s} n={len(idx):5d}  {hit[idx].mean()*100:5.1f}%")
    ho_hit = {}
    print("\n=== 호별 적중률 (n<30 참고용) ===")
    for ho in sorted({h for r in scored_rows for h in r["hos"]}):
        idx = [k for k, r in enumerate(scored_rows) if ho in r["hos"]]
        ho_hit[ho] = [float(hit[idx].mean()), len(idx)]
        lo_, hi_ = wilson(int(hit[idx].sum()), len(idx))
        print(f"  {ho:28s} n={len(idx):5d}  {hit[idx].mean()*100:5.1f}% [{lo_*100:.0f}~{hi_*100:.0f}%]{'' if len(idx) >= 30 else '  측정 불가(호 n<30 · D-40)'}")

    # ══════════════════════════════════════════════════════════════════════════════════════════
    # 🆕 v8.2 (D-321) **8종 기준 표** — 위(6종 기준)와 같은 모델 · 같은 문턱이다. 채점 행과 플래그 범위만 넓힌다
    # ══════════════════════════════════════════════════════════════════════════════════════════
    th8 = np.array([TH[l] for l in CONF8])
    rows8 = BUCKETS["scored8"]
    y8 = np.stack([to_multihot(r["labels"]) for r in rows8])[:, :N8].astype(int)
    p8 = predict_probs(best_model, rows8)[:, :N8]
    pred8, pred8_05 = (p8 >= th8).astype(int), (p8 > 0.5).astype(int)
    print(f"\n\n[test **8종 기준** · {len(rows8)}행 = 6종 기준 {len(scored_rows)} + 부당 비교 · 비방이 붙은 행 {len(rows8) - len(scored_rows)} · 시드 {BEST_SEED}]")
    print(f"{'유형':20s} {'정답':>5s} {'th':>6s} {'P':>6s} {'R':>6s} {'F1':>6s} {'F1@0.5':>7s} {'ROC':>6s} {'PR-AUC':>7s}")
    test8_label = {}
    for i, l in enumerate(CONF8):
        n = int(y8[:, i].sum())
        p_, r_, f_, _ = precision_recall_fscore_support(y8[:, i], pred8[:, i], average="binary", zero_division=0)
        f05 = f1_score(y8[:, i], pred8_05[:, i], zero_division=0)
        roc, pr = safe(roc_auc_score, y8[:, i], p8[:, i]), safe(average_precision_score, y8[:, i], p8[:, i])
        tp_ = int((pred8[:, i] & y8[:, i]).sum()); pp_ = int(pred8[:, i].sum())
        rci, pci = wilson(tp_, n), wilson(tp_, pp_)
        test8_label[l] = {"n": n, "th": TH[l], "P": p_, "R": r_, "F1": f_, "F1@0.5": f05, "rocauc": roc, "prauc": pr, "R_ci": rci, "P_ci": pci, "k": tp_, "pp": pp_}
        fmt = lambda v: f"{v:6.3f}" if v is not None else "   N/A"
        print(f"{l:20s} {n:5d} {TH[l]:6.3f} {p_:6.3f} {r_:6.3f} {f_:6.3f} {f05:7.3f} {fmt(roc)} {fmt(pr):>7s}{'  측정 불가(n<30)' if n < 30 else ''}"
              f"  R CI [{rci[0]:.2f},{rci[1]:.2f}] P CI [{pci[0]:.2f},{pci[1]:.2f}]{'  🆕' if l in NEW_CONF else ''}")
    _, _, micro8, _ = precision_recall_fscore_support(y8, pred8, average="micro", zero_division=0)
    _, _, micro8_05, _ = precision_recall_fscore_support(y8, pred8_05, average="micro", zero_division=0)
    meas8 = [l for l in CONF8 if test8_label[l]["n"] >= 30]
    macro8 = float(np.mean([test8_label[l]["F1"] for l in meas8])) if meas8 else float("nan")
    print(f"\nmicro F1 {micro8:.3f} (0.5 고정 {micro8_05:.3f}) · 측정 가능 {len(meas8)}종 macro F1 {macro8:.3f}   ← 6종 기준: micro F1 {micro_dev:.3f} · 측정 가능 {len(measurable)}종 macro F1 {macro_meas:.3f}")

    # ── 비방 ↔ 부당 비교를 바꿔 낸 수 (팀장 10-06 — 식품 6호 · 7호는 경계가 겹친다)
    print("\n=== 비방 ↔ 부당 비교를 바꿔 낸 수 (정답이 한쪽뿐인 행 · 8종 기준 채점 행) ===")
    test_swap = swap_counts(y8, pred8)
    print_swap(test_swap)
    _pair = y8[:, [CONF8.index(l) for l in NEW_CONF]].any(1)
    _pair_hit = pred8[:, [CONF8.index(l) for l in NEW_CONF]].any(1)
    if _pair.any():
        _lo, _hi = wilson(int(_pair_hit[_pair].sum()), int(_pair.sum()))
        print(f"  묶음 재현율 (비방 · 부당 비교 중 하나라도 정답 → 둘 중 하나라도 냄) {int(_pair_hit[_pair].sum())}/{int(_pair.sum())} {_pair_hit[_pair].mean() * 100:.1f}% [{_lo * 100:.1f}~{_hi * 100:.1f}%]")

    # ── 6종 기준 ↔ 8종 기준 — 조건 버킷을 나란히 (같은 행 · 플래그 범위만 다르다)
    bp_all = {b: predict_probs(best_model, BUCKETS[b]) for b in ["neg", "D", "M", "pending", "대기"]}
    _k6 = lambda p: int((p[:, :N_CONF] >= th_vec).any(axis=1).sum()) if len(p) else 0
    _k8 = lambda p: int((p[:, :N8] >= th8).any(axis=1).sum()) if len(p) else 0
    _gf8 = np.array([r["id"].startswith("gf:") for r in BUCKETS["D"]], dtype=bool)
    cmp8 = {
        "양성 재현율 (유형 있는 위반)": [int(pred_dev.any(axis=1).sum()), len(scored_rows), int(pred8.any(axis=1).sum()), len(rows8)],
        "  그중 정답 유형을 맞힘": [int((pred_dev & y_te).any(axis=1).sum()), len(scored_rows), int((pred8 & y8).any(axis=1).sum()), len(rows8)],
        "음성 오판정 (조건 L)": [_k6(bp_all["neg"]), len(BUCKETS["neg"]), _k8(bp_all["neg"]), len(BUCKETS["neg"])],
        "적법 · 주장 없음 (수정문구 D)": [_k6(bp_all["D"][_gf8]), int(_gf8.sum()), _k8(bp_all["D"][_gf8]), int(_gf8.sum())],
        "참고 — 해설서 위반문구 D": [_k6(bp_all["D"][~_gf8]), int((~_gf8).sum()), _k8(bp_all["D"][~_gf8]), int((~_gf8).sum())],
        "참고 — 조건 M (채점 안 함)": [_k6(bp_all["M"]), len(BUCKETS["M"]), _k8(bp_all["M"]), len(BUCKETS["M"])],
        "따로 — 유형 없는 위반": [_k6(bp_all["pending"]), len(BUCKETS["pending"]), _k8(bp_all["pending"]), len(BUCKETS["pending"])],
        "따로 — 편입 대기 유형만 있는 행": [_k6(bp_all["대기"]), len(BUCKETS["대기"]), _k8(bp_all["대기"]), len(BUCKETS["대기"])],
    }
    def _cell(k, n):
        if not n:
            return f"{'행 없음':>30s}"
        lo_, hi_ = wilson(k, n)
        return f"{k:5d}/{n:<5d} {k / n * 100:5.1f}% [{lo_ * 100:4.1f}~{hi_ * 100:4.1f}]"
    print("\n=== 6종 기준 ↔ 8종 기준 (dev threshold · 하나라도 울린 행 · Wilson 95%) ===")
    print(f"{'':30s} {'6종 기준':>30s}   {'8종 기준':>30s}")
    for name, (k6, n6, k8_, n8_) in cmp8.items():
        print(f"{name:30s} {_cell(k6, n6)}   {_cell(k8_, n8_)}{'  측정 불가(n<30)' if 0 < min(n6, n8_) < 30 else ''}")
    print("  · 양성 재현율은 두 표의 채점 행이 다르다(8종 기준이 더 많다). 음성 · D · M · 따로 줄은 같은 행이고 8종 쪽이 같거나 많다")
    print("  · 유형 없는 위반 · 편입 대기는 양성 재현율에 넣지 않는다 — 인코더가 그 유형을 내지 않는다(다른 유형으로 울린 수다)")

    # ── 편입 대기 · 기능성화장품_오인 — 0.5 기준 참고값 (팀장 10-06). 조건 M 을 뺀 평가 행 전부에서 잰다
    _ev = [r for r in eval_rows if r["bucket"] != "M"]
    _pw = predict_probs(best_model, _ev)[:, IDX_WAIT]
    _yw = np.array([WAITING[0] in r["labels"] for r in _ev], dtype=int)
    _wk, _wn, _wpp = int(((_pw >= WAIT_REF_TH) & (_yw == 1)).sum()), int(_yw.sum()), int((_pw >= WAIT_REF_TH).sum())
    _wr, _wp = wilson(_wk, _wn), wilson(_wk, _wpp)
    test_wait = {"label": WAITING[0], "ref_threshold": WAIT_REF_TH, "n": _wn, "k": _wk, "pp": _wpp, "rows": len(_ev),
                 "prauc": safe(average_precision_score, _yw, _pw), "rocauc": safe(roc_auc_score, _yw, _pw)}
    print(f"\n=== 편입 대기 · {WAITING[0]} — {WAIT_REF_TH} 기준 참고값 (저장 문턱은 {TH_OFF} 로 꺼 둔다 · 조건 M 제외 {len(_ev)}행) ===")
    print(f"  정답 n={_wn}  R {_wk}/{_wn} {_wk / max(_wn, 1) * 100:.1f}% [{_wr[0] * 100:.0f}~{_wr[1] * 100:.0f}%]  P {_wk}/{_wpp} {_wk / max(_wpp, 1) * 100:.1f}% [{_wp[0] * 100:.0f}~{_wp[1] * 100:.0f}%]"
          f"  PR-AUC {test_wait['prauc'] if test_wait['prauc'] is None else round(test_wait['prauc'], 3)}" + ("  측정 불가(n<30)" if _wn < 30 else "") + "  · 학습 양성 "
          + str(TRAIN_INJ[WAITING[0]][0]) + "행")

    # ── 거짓_과장 따로 보기 (팀장 10-06 — 낱말 제외에서 dev 재현율이 3.9%p 내렸다). 6종 기준 채점 행
    _i4 = CONFIRMED.index("거짓_과장")
    _g = y_te[:, _i4] == 1
    _gk, _gn = int(pred_dev[_g, _i4].sum()), int(_g.sum())
    _glo, _ghi = wilson(_gk, _gn)
    falsity = {"n": _gn, "k": _gk, "R": _gk / max(_gn, 1), "R_ci": [_glo, _ghi], "P": test_label["거짓_과장"]["P"], "th": TH["거짓_과장"],
               "R_at05": float(pred_05[_g, _i4].mean()) if _gn else float("nan"),
               "dev_R": dev_cond.get("R:거짓_과장"), "dev_n": dev_cond.get("n_R:거짓_과장"), "by_ho": {}, "by_cond": {}}
    print(f"\n=== 거짓_과장 따로 보기 (6종 기준 채점 행 · 문턱 {TH['거짓_과장']:.3f}) ===")
    print(f"  test 재현율 {_gk}/{_gn} {falsity['R'] * 100:.1f}% [{_glo * 100:.1f}~{_ghi * 100:.1f}%] · 정밀도 {falsity['P'] * 100:.1f}% · 0.5 기준 재현율 {falsity['R_at05'] * 100:.1f}%"
          f"   ← dev 재현율 {falsity['dev_R'] * 100:.1f}% (n={falsity['dev_n']}) · dev 는 train 과 같은 원천이라 높게 나온다")
    print("  호별 (그 호가 근거에 있는 거짓_과장 정답 행)")
    for ho in sorted({h for k_, r in enumerate(scored_rows) if _g[k_] for h in r["hos"]}):
        idx = [k_ for k_, r in enumerate(scored_rows) if _g[k_] and ho in r["hos"]]
        kk = int(pred_dev[idx, _i4].sum()); lo_, hi_ = wilson(kk, len(idx))
        falsity["by_ho"][ho] = [kk, len(idx)]
        print(f"    {ho:28s} {kk:4d}/{len(idx):<4d} {kk / len(idx) * 100:5.1f}% [{lo_ * 100:.0f}~{hi_ * 100:.0f}%]{'' if len(idx) >= 30 else '  측정 불가(n<30)'}")
    print("  조건별")
    for c, name in [("C", "C 표현 자체"), ("A", "A 조성·지위"), ("B", "B 실증 필요"), (None, "조건 없음")]:
        idx = [k_ for k_, r in enumerate(scored_rows) if _g[k_] and r["condition"] == c]
        if idx:
            kk = int(pred_dev[idx, _i4].sum()); lo_, hi_ = wilson(kk, len(idx))
            falsity["by_cond"][name] = [kk, len(idx)]
            print(f"    {name:16s} {kk:4d}/{len(idx):<4d} {kk / len(idx) * 100:5.1f}% [{lo_ * 100:.0f}~{hi_ * 100:.0f}%]{'' if len(idx) >= 30 else '  측정 불가(n<30)'}")

    test8 = {"micro_f1": micro8, "micro_f1_at05": micro8_05, "macro_f1_measurable": macro8, "label": test8_label, "swap": test_swap,
             "cmp_6_vs_8": cmp8, "waiting": test_wait, "falsity": falsity, "n_scored8": len(rows8)}

    # ══ 🆕 v9 — 더한 표 (채점 규칙은 그대로 · 같은 확률을 다르게 묶어 본다)
    # ── (가) dev → test — 6종 기준 · 같은 모델 · 같은 문턱. dev 는 train 과 같은 원천이라 높게 나온다
    print("\n\n=== (가) dev → test · 6종 기준 (같은 모델 · 같은 문턱) ===")
    print(f"{'':22s} {'dev':>20s}   {'test':>20s}   {'차이':>8s}")
    dev_vs_test = {}
    def _dt(name, dk, dn, tk, tn):
        dev_vs_test[name] = [dk, dn, tk, tn]
        d = f"{dk:4d}/{dn:<4d} {dk / dn * 100:5.1f}%" if dn else f"{'—':>16s}"
        t = f"{tk:4d}/{tn:<4d} {tk / tn * 100:5.1f}%" if tn else f"{'—':>16s}"
        gap = f"{(tk / tn - dk / dn) * 100:+7.1f}%p" if dn and tn else ""
        print(f"{name:22s} {d:>20s}   {t:>20s}   {gap}" + ("  측정 불가(dev 또는 test n<30)" if 0 < min(dn, tn) < 30 else ""))
    _dt("위반 재현율 (하나라도)", dev_cond["k_pos_hit"], dev_cond["n_pos"], _pos_k, _pos_n)
    for i, l in enumerate(CONFIRMED):
        _dt(l, dev_cond.get(f"k_R:{l}", 0), dev_cond.get(f"n_R:{l}", 0), int((pred_dev[:, i] & y_te[:, i]).sum()), int(y_te[:, i].sum()))
    _dt("오판정 · 조건 L", dev_cond.get("k_neg_flag:L", 0), dev_cond.get("n_neg:L", 0), _neg_k, _neg_n)
    _dt("울림 · 조건 D (전부)", dev_cond.get("k_neg_flag:D", 0), dev_cond.get("n_neg:D", 0), _d_k, _d_n)
    print("  · dev 의 조건 D 는 학습 분할의 D 이고 test 의 D 는 수정문구 판과 해설서 위반문구가 섞여 있다(D-301) — 구성이 달라 참고로만 본다")

    # ── (나) 8종 기준 — 호별 · 법별 (팀장 시험지: 호 단위)
    hit8 = (pred8 & y8).any(axis=1)
    any8 = pred8.any(axis=1)
    ho_hit8, law8 = {}, {}
    print("\n=== (나) 호별 · 8종 기준 (정답 유형을 맞힘 / 하나라도 울림 · n<30 참고용) ===")
    for ho in sorted({h for r in rows8 for h in r["hos"]}):
        idx = [k for k, r in enumerate(rows8) if ho in r["hos"]]
        kh, ka = int(hit8[idx].sum()), int(any8[idx].sum())
        ho_hit8[ho] = [kh, len(idx), ka]
        lo_, hi_ = wilson(kh, len(idx))
        print(f"  {ho:28s} n={len(idx):5d}  맞힘 {kh / len(idx) * 100:5.1f}% [{lo_ * 100:.0f}~{hi_ * 100:.0f}%] · 울림 {ka / len(idx) * 100:5.1f}%"
              f"{'' if len(idx) >= 30 else '  측정 불가(호 n<30 · D-40)'}{'  ← 6종 기준 표에 없던 호' if ho not in ho_hit else ''}")
    print("  법별 · 8종 기준")
    for law in sorted({lw for r in rows8 for lw in r["laws"]}):
        idx = [k for k, r in enumerate(rows8) if law in r["laws"]]
        _, _, f, _ = precision_recall_fscore_support(y8[idx], pred8[idx], average="micro", zero_division=0)
        law8[law] = {"n": len(idx), "micro_f1": float(f), "k_hit": int(hit8[idx].sum()), "k_any": int(any8[idx].sum())}
        print(f"    {LAW_NAMES.get(law, law):12s} n={len(idx):5d}  micro F1={f:.3f} · 맞힘 {hit8[idx].mean() * 100:5.1f}% · 울림 {any8[idx].mean() * 100:5.1f}%")

    P_EV = predict_probs(best_model, eval_rows)          # 평가 행 전부의 9칸 확률 (행별 csv)
    _evb = np.array([r["bucket"] for r in eval_rows])
    OUT_NAME, dev_p = RUN_NAME, seed_results[BEST_SEED]["dev_p"]
    # ── (라) 행별 확률 — 문구는 싣지 않는다. 보고서의 추가 표는 이 파일로 만든다 (test 를 다시 돌리지 않는다)
    def write_probs(path, rows, probs, extra_cols):
        with open(path, "w", newline="", encoding="utf-8-sig") as f:
            w = csv.writer(f)
            w.writerow(["id", "조건", "정답유형", "근거호", "법", "provenance", "unit"] + [c for c, _ in extra_cols]
                       + [f"p_{l}" for l in LABEL_LIST] + ["flag6", "flag8"])
            for r, p in zip(rows, probs):
                w.writerow([r["id"], r["condition"] or "", "|".join(r["labels"]), "|".join(r["hos"]), "|".join(r["laws"]), r["provenance"] or "", r["unit"] or ""]
                           + [fn(r) for _, fn in extra_cols] + [f"{float(v):.6f}" for v in p]
                           + [int((p[:N_CONF] >= th_vec).any()), int((p[:N8] >= th8).any())])
    CSV_TEST, CSV_DEV = OUT_NAME + "_행별확률.csv", OUT_NAME + "_dev행별확률.csv"
    write_probs(CSV_TEST, eval_rows, P_EV, [("split", lambda r: r["split"]), ("bucket6", lambda r: r["bucket"]), ("bucket8", lambda r: r["bucket8"]),
                                            ("적법구분", lambda r: lawful_kind(r) or "")])
    write_probs(CSV_DEV, dev_rows, dev_p, [("주입", lambda r: int(is_inj(r)))])
    print(f"\n행별 확률: {CSV_TEST} ({len(eval_rows)}행) · {CSV_DEV} ({len(dev_rows)}행) — 문구 없음 · flag 는 적용 문턱 기준")
    # 행별 파일과 위 표가 같은 수를 내는가 (같은 확률을 두 번 냈다 — 묶음이 달라도 판정이 같아야 한다)
    _chk = int((P_EV[_evb == "scored"][:, :N_CONF] >= th_vec).any(axis=1).sum())
    assert _chk == _pos_k, f"행별 확률과 6종 기준 표가 다르다 ({_chk} · {_pos_k})"
    test8["extra"] = {"dev_vs_test": dev_vs_test, "ho_hit8": ho_hit8, "law8": law8, "threshold_mode": THRESHOLD_MODE,
                      "thresholds_best_seed": dict(TH_BEST), "thresholds_robust": dict(TH_ROBUST)}


## 8. 기준선 B(사전 단독판정) + G1 게이트 — 확정 8종

🆕 v8.2 — 기존 6종은 6종 기준 채점 행(`scored`)에서 · 새 2종은 8종 기준 채점 행(`scored8`)에서 잰다.

G1: 모델 Recall(dev threshold) ≥ 기준선 B Recall + 0.15. 정답 30건 미만은 참고용(D-40).
🔄 v8 — 사전이 506종이 되고 단독판정 자격이 줄었다(D-311 `비주장문맥`). 기준선 B 는 올린 사전으로 다시 잡힌다 — 사전 sha 를 로그에 남긴다.
기준선은 `원문 in text`(정규화 없음)라 판정기 B(`app/dictmatch.py`)와 수가 조금 다를 수 있다.


In [ ]:
if FINAL_TEST:
    banned_terms_rows = load_jsonl("banned_terms.jsonl")
    solo = [r for r in banned_terms_rows if r.get("단독판정") is True]
    term_to_labels = [(s, set(r.get("유형", [])) & set(CONF8)) for r in solo for s in r.get("원문", [])]
    print(f"banned_terms {len(banned_terms_rows)}건 · 단독판정 {len(solo)}건 · 비주장문맥 {sum(1 for r in banned_terms_rows if r.get('신뢰도') == '비주장문맥')}건 · sha {BANNED_SHA[:12]}")

    def rule_predict(text):
        out = set()
        for s, ls in term_to_labels:
            if s and s in text:
                out |= ls
        return out

    rule_preds = [rule_predict(r["text"]) for r in scored_rows]
    GATE_MARGIN, MIN_N = 0.15, 30
    gate_rows = []
    print(f"\n{'유형':20s} {'기준선B':>8s} {'모델R':>7s} {'통과선':>7s} {'n':>5s}  판정")
    for i, l in enumerate(CONFIRMED):
        idx = [k for k in range(len(scored_rows)) if y_te[k, i]]
        n = len(idx)
        if not n:
            continue
        b = sum(l in rule_preds[k] for k in idx) / n
        m = float(pred_dev[idx, i].mean())
        verdict = f"참고용(n<{MIN_N})" if n < MIN_N else ("통과" if m >= b + GATE_MARGIN else "미달")
        gate_rows.append({"label": l, "baseline_recall": b, "model_recall": m, "pass_line": b + GATE_MARGIN, "n": n, "verdict": verdict})
        print(f"{l:20s} {b:8.3f} {m:7.3f} {b + GATE_MARGIN:7.3f} {n:5d}  {verdict}")
    # 🆕 v8.2 — 새 확정 2종 (8종 기준 채점 행)
    rule_preds8 = [rule_predict(r["text"]) for r in rows8]
    for l in NEW_CONF:
        i = CONF8.index(l)
        idx = [k for k in range(len(rows8)) if y8[k, i]]
        n = len(idx)
        if not n:
            continue
        b = sum(l in rule_preds8[k] for k in idx) / n
        m = float(pred8[idx, i].mean())
        verdict = f"참고용(n<{MIN_N})" if n < MIN_N else ("통과" if m >= b + GATE_MARGIN else "미달")
        gate_rows.append({"label": l, "baseline_recall": b, "model_recall": m, "pass_line": b + GATE_MARGIN, "n": n, "verdict": verdict, "rows": "scored8"})
        print(f"{l:20s} {b:8.3f} {m:7.3f} {b + GATE_MARGIN:7.3f} {n:5d}  {verdict}  🆕")
    _miss = [g["label"] for g in gate_rows if g["verdict"] == "미달"]
    print("\n사전 고정 test 기준(머리말) —", "전부 통과" if not _miss else f"미달 {_miss} — 「미달」로 보고한다. 이 결과로 설정을 다시 고르지 않는다 (D-175)")
else:
    print("🔒 기준선 B 게이트는 최종 측정 때 함께 본다 (test 사용 · D-175)")


## 9. 실험 누적 — 후보 비교는 dev 평균 ± 표준편차로

표의 dev 열로 후보를 고른다. Drive에 `copylane_v8_experiment_log.json`으로 누적 저장한다(v7 로그와 평가셋이 달라 파일을 나눴다).
비교 기준은 `all · 낱말 포함`(머리말의 사전 고정 기준) — 그 실행이 로그에 있어야 자동 판정이 나온다.


🆕 v8.2 — 비교표 아래에 **재현 확인**이 붙는다. 학습 행 · dev 행 · 선택 기준이 같으므로 6종 dev 수치가 2회차(6종 저장판)와 같아야 한다. 후보 비교 · 사전 고정 기준(G1 ~ G3 · B2)은 지금도 **6종 dev** 로 잰다.

🆕 v9 — 비교 기준은 **R0(v8.3 기준 실행 · best_seed 문턱)** 이다. 같은 학습이므로 PR-AUC 는 같아야 하고(「R0 재현」), G1 ~ G3 은 문턱 규칙을 바꾼 효과를 보여 준다.


In [ ]:
_before = EXPERIMENT_LOG.get(exp_key)   # 🆕 v8.2 — 같은 키의 앞선 실행(A 회차) · 재현 확인에 쓴다
EXPERIMENT_LOG[exp_key] = {
    "golden_sha256": GOLDEN_SHA, "extra_train": EXTRA_TRAIN_FILES, "eval_only": EVAL_ONLY_FILES, "extra_sha256": EXTRA_SHA,
    "extra_rows": len(extra_rows), "extra_holdout_rows": {f: len(r) for f, r in extra_holdout.items()}, "seeds": SEEDS, "best_seed": BEST_SEED,
    "dev_macro_prauc_mean": float(np.mean(devs)), "dev_macro_prauc_std": float(np.std(devs)),
    "dev_by_seed": {str(s): seed_results[s]["dev"] for s in SEEDS}, "dev_cond": dev_cond,
    "best_epoch_by_seed": {str(s): seed_results[s]["best_epoch"] for s in SEEDS},
    "thresholds": TH, "train_rows": len(train_rows), "dev_rows": len(dev_rows), "notebook": "v9", "final_test": FINAL_TEST, "threshold_mode": THRESHOLD_MODE,
    "thresholds_best_seed": seed_results[BEST_SEED]["thresholds"], "thresholds_robust": TH_ROBUST, "dev_by_rule": DEV_RULES,
    "test_scoring_note": "이 골든 판 test 의 5번째 채점(탐색에서 R0 을 쟀다면 6번째) · dev 정의 · 문턱 규칙은 후보 A 의 test 결과를 본 뒤 바꿨다 · 문턱 값은 dev 에서만 나왔다",
    "dev_sha": DEV_SHA, "dev_score_injected": DEV_SCORE_INJECTED, "dev_stratify_new": DEV_STRATIFY_NEW, "dev_inj_rows": len(DEV_INJ),
    "inj_weight": INJ_WEIGHT, "max_epochs": MAX_EPOCHS, "use_injected": USE_INJECTED,
    "confirmed": CONF8, "waiting": WAITING, "selection": "기존 6종 dev macro PR-AUC (회차 · 시드)", "dev8": dev8, "train_injected": TRAIN_INJ,
    "label_unit": LABEL_UNIT, "train_labels": list(TRAIN_LABELS),
    "banned_terms_sha256": BANNED_SHA, "use_word_rows": USE_WORD_ROWS, "drop_untyped_pos": DROP_UNTYPED_POS,
    "train_by_cond": TRAIN_BY_COND, "train_role_counts": dict(ROLE_COUNTS),
}
if FINAL_TEST:
    EXPERIMENT_LOG[exp_key]["test"] = {"micro_f1": micro_dev, "micro_f1_at05": micro_05, "macro_f1_measurable": macro_meas,
        "f1_5ho": f5ho, "macro_prauc": tm["macro_prauc"], "macro_rocauc": tm["macro_rocauc"],
        **bucket_summary, "law_f1": law_f1, "label": test_label, "ho_hit": ho_hit, "gate": gate_rows}
    EXPERIMENT_LOG[exp_key]["test8"] = test8   # 🆕 v8.2 — 8종 기준 표 · 바꿔 낸 수 · 편입 대기 · 거짓_과장

pct = lambda v: f"{v*100:5.1f}%" if isinstance(v, (int, float)) and v == v else "    —"
def same_dev(m):
    # 🆕 v8.3 — 같은 dev 행으로 잰 실행인가. v8 ~ v8.2 로그에는 지문이 없다 → 옛 dev 정의로 돌릴 때만 같은 것으로 본다
    return m.get("dev_sha") == DEV_SHA or (DEV_LEGACY and "dev_sha" not in m)
hold_cols = sorted({k for m in EXPERIMENT_LOG.values() for k in m.get("dev_cond", {}) if k.startswith("hold:")})
short = lambda k: k.split(":", 1)[1].replace("proto_", "")
print(f"| {'실험':58s} | {'dev PR-AUC':>13s} | {'dev 재현율':>8s} | {'dev 음성FPR':>9s} | " + "".join(f"{short(c):>16s} | " for c in hold_cols) + "비고 |")
for k, m in sorted(EXPERIMENT_LOG.items(), key=lambda kv: -kv[1]["dev_macro_prauc_mean"]):
    dc = m.get("dev_cond", {})
    note = "최종 test 채점" if m.get("final_test") else ("탐색치(D-175 이전 · test 봤음)" if "test" in m else "")
    note += "" if same_dev(m) else " dev 가 다르다(견주지 않는다)"
    print(f"| {k:58s} | {m['dev_macro_prauc_mean']:.3f} ± {m['dev_macro_prauc_std']:.3f} | {pct(dc.get('dev_violation_recall')):>8s} | "
          f"{pct(dc.get('dev_neg_fpr')):>9s} | " + "".join(f"{pct(dc.get(c)):>16s} | " for c in hold_cols) + f"{note} |")
print("(홀드아웃 칸: 탐지율은 높을수록 · 오탐률은 낮을수록 좋다. 같은 행으로 잰 수라 실행끼리 비교된다)")

# ── v8 후보 비교 — 같은 골든 · 같은 백본끼리만. 기준은 all · 낱말 포함(BASE_KEY)
def _wil(k, n):
    lo, hi = wilson(k, n)
    return f"{k / n * 100:5.1f}% [{lo * 100:4.1f}~{hi * 100:4.1f}]" if n else "    —"

def cand_name(k):
    # 'kcbert|이유=all|focal(…)|+proto…|v8|낱말제외|호학습' → 'all·낱말제외·호학습'
    tail = k.split("|v8", 1)[1] if "|v8" in k else ""
    return k.split("|")[1].replace("이유=", "") + tail.replace("|", "·")

CANDS = {k: m for k, m in EXPERIMENT_LOG.items()
         if k.startswith(f"{BACKBONE}|이유=") and str(m.get("notebook", "")).startswith(("v8", "v9")) and m.get("golden_sha256") == GOLDEN_SHA
         and same_dev(m)}   # 🔄 v8.3 — 같은 dev 행으로 잰 실행끼리만
print(f"\n=== v8 후보 비교 ({len(CANDS)}건 · 기준 {cand_name(BASE_KEY) if BASE_KEY in CANDS else '없음 — 기준 실행(' + cand_name(BASE_KEY) + ')을 먼저 돌린다'}) ===")
print(f"{'유형별 dev 재현율 (Wilson 95%)':34s} " + " | ".join(f"{cand_name(k):>24s}" for k in CANDS))
for l in CONFIRMED:
    row = []
    for k, m in CANDS.items():
        dc = m.get("dev_cond", {})
        row.append(_wil(dc.get(f"k_R:{l}", 0), dc.get(f"n_R:{l}", 0)) if f"n_R:{l}" in dc else "    —")
    n_ = next((m["dev_cond"].get(f"n_R:{l}") for m in CANDS.values() if f"n_R:{l}" in m.get("dev_cond", {})), 0)
    print(f"{l:20s} n={n_:<4d}{'(측정 불가)' if n_ < 30 else '          '} " + " | ".join(f"{c:>24s}" for c in row))
# 🆕 v8.1 — 유형 없는 위반(dev 묶음 · 학습 안 함) 탐지율 · 참고
_ur = [_wil(m["dev_cond"]["k_untyped_hit"], m["dev_cond"]["n_untyped"]) if m.get("dev_cond", {}).get("n_untyped") else "    —" for m in CANDS.values()]
print(f"{'유형 없는 위반 탐지 (참고)':34s} " + " | ".join(f"{c:>24s}" for c in _ur))

# 🆕 v8.2 재현 확인 — 학습 행 · dev 행 · 선택 기준이 같으므로 **6종 dev 수치가 같은 설정의 앞선 실행과 같아야** 한다
#    (GPU 연산 순서 때문에 소수 셋째 자리는 달라질 수 있다. 크게 다르면 스위치 · 올린 파일을 확인한다)
def _repro(name, m):
    d, me = m.get("dev_cond", {}), EXPERIMENT_LOG[exp_key]
    same_th = all(abs(m.get("thresholds", {}).get(l, -1) - TH[l]) < 1e-9 for l in CONFIRMED)
    print(f"  {name:34s} PR-AUC {m['dev_macro_prauc_mean']:.3f} ± {m['dev_macro_prauc_std']:.3f} · 재현율 {d.get('k_pos_hit', '?')}/{d.get('n_pos', '?')} · "
          f"음성 오판정 {d.get('k_neg_flag', '?')}/{d.get('n_neg', '?')} · 시드 {m.get('best_seed')} · 6종 문턱 {'같다' if same_th else '다르다'}")
    gap = abs(m["dev_macro_prauc_mean"] - me["dev_macro_prauc_mean"])
    drift = (d.get("k_pos_hit") != dev_cond["k_pos_hit"]) or (d.get("k_neg_flag") != dev_cond["k_neg_flag"])
    print("    → " + ("✅ 같다" if gap < 0.0015 and not drift and same_th else
                      ("🟡 조금 다르다 — 다시 학습해 가중치가 달라졌다(GPU 비결정성 범위인지 본다)" if gap < 0.01 else "🔴 다르다 — 스위치 · 파일 · 학습 행 수를 확인한다")))
print(f"\n=== 재현 확인 (6종 dev · 같은 학습 설정이면 같아야 한다) ===")
if not DEV_LEGACY:
    print("  dev 정의가 v8.2 까지와 다르다(실제 문구만 · 8종 층화) — 2회차 · v8.2 수치와 견주지 않는다. 같은 설정의 앞선 실행만 본다")
print(f"  {'이번 실행' + (' (8종 문턱 · 9칸 저장)' if LABEL_UNIT == 'type' else ' (호 판)'):34s} PR-AUC {np.mean(devs):.3f} ± {np.std(devs):.3f} · 재현율 {dev_cond['k_pos_hit']}/{dev_cond['n_pos']} · "
      f"음성 오판정 {dev_cond['k_neg_flag']}/{dev_cond['n_neg']} · 시드 {BEST_SEED}" + ("   ♻️ 학습한 시드를 그대로 썼다" if REUSE else ""))
if DEV_LEGACY:
    if PREV_KEY in EXPERIMENT_LOG and PREV_KEY != exp_key:
        _repro("6종 저장판 (v8 · v8.1 같은 설정)", EXPERIMENT_LOG[PREV_KEY])
    else:
        print("  6종 저장판(v8 · v8.1 같은 설정) 로그가 없다")
if _before is not None:
    _repro("이 설정의 앞선 실행 (A 회차)", _before)

if BASE_KEY not in CANDS and BASE_KEY_ALT in CANDS:   # 호 판 — 8종 저장판 유형 실행이 로그에 없으면 6종 저장판으로 견준다(6종 dev 수치는 같다)
    BASE_KEY = BASE_KEY_ALT
if BASE_KEY in CANDS:
    b = CANDS[BASE_KEY]["dev_cond"]; b_au = CANDS[BASE_KEY]["dev_macro_prauc_mean"]
    hk = next((c for c in b if c.startswith("hold:") and c.endswith(":탐지율")), None)
    print(f"\n=== 사전 고정 기준 (이번 실행 ↔ 기준 {cand_name(BASE_KEY)} · G5 는 로컬 run_judge_dist 로 본다) ===")
    for k, m in CANDS.items():
        if k != exp_key or k == BASE_KEY:   # 🔄 v8.1 — 이번 실행만 그 기준과 견준다
            continue
        d = m["dev_cond"]; res = []
        g1 = (b["dev_violation_recall"] - d["dev_violation_recall"]) * 100
        res.append(("G1 재현율 하락 ≤ 2%p", g1 <= 2.0, f"하락 {g1:.1f}%p ({b.get('k_pos_hit', '?')}→{d.get('k_pos_hit', '?')}줄)"))
        for l in CONFIRMED:   # v8 — 기준 실행의 dev 양성이 30 이상인 유형만 (D-40)
            if b.get(f"n_R:{l}", 0) >= 30 and f"R:{l}" in d:
                g2 = (b[f"R:{l}"] - d[f"R:{l}"]) * 100
                res.append((f"G2 {l} 하락 ≤ 5%p", g2 <= 5.0, f"하락 {g2:.1f}%p"))
        g3 = d.get("k_neg_flag", 0) - b.get("k_neg_flag", 0)
        res.append(("G3 적법 오판정 증가 ≤ 1줄", g3 <= 1, f"증가 {g3:+d}줄"))
        if hk and hk in d:
            g4 = (b[hk] - d[hk]) * 100
            res.append(("G4 4호 탐지 하락 ≤ 5%p", g4 <= 5.0, f"하락 {g4:.1f}%p"))
        d8, b8 = m.get("dev8", {}), CANDS[BASE_KEY].get("dev8", {})
        if "k_neg_flag8" in d8 and "k_neg_flag8" in b8:   # 🆕 v8.3 — 음성 오판정을 8종 기준으로도
            g38 = d8["k_neg_flag8"] - b8["k_neg_flag8"]
            res.append(("G3 적법 오판정(8종) 증가 ≤ 1줄", g38 <= 1, f"증가 {g38:+d}줄"))
        b2 = (b_au - m["dev_macro_prauc_mean"])
        res.append(("B2 PR-AUC 하락 ≤ 0.02", b2 <= 0.02, f"변화 {-b2:+.3f}"))
        ok = all(r[1] for r in res)
        print(f"\n[{cand_name(k)}] → {'탈락 기준 통과 (G5 로컬 확인)' if ok else '탈락'}")
        for name, passed, val in res:
            print(f"  {'✅' if passed else '❌'} {name:26s} {val}")
        if LABEL_UNIT == "type" and THRESHOLD_MODE != "robust":   # 🆕 v8.3 — 채택 조건(머리말 · 돌리기 전에 정했다): 6종 PR-AUC +0.008 이상 또는 새 2종 PR-AUC 평균 +0.05 이상
            _gain = m["dev_macro_prauc_mean"] - b_au
            _a2, _b2 = d8.get("new2_prauc_mean"), b8.get("new2_prauc_mean")
            _gain2 = (_a2 - _b2) if (_a2 is not None and _b2 is not None) else None
            _g2s = "N/A" if _gain2 is None else f"{_gain2:+.3f}"
            _pick = ok and (_gain >= 0.008 or (_gain2 is not None and _gain2 >= 0.05))
            print(f"  채택 조건 — 6종 PR-AUC {_gain:+.3f} (≥ +0.008) · 새 2종 PR-AUC 평균 {_g2s} (≥ +0.05) → {'채택 후보' if _pick else '기준(R0) 유지'}")
            for l in NEW_CONF:   # 새 2종의 실제 문구 재현율 (참고 — 줄 수)
                _x, _y = b8.get(f"{l}|전체"), d8.get(f"{l}|전체")
                if _x and _y:
                    print(f"    {l}: dev 재현율 {_x['k']}/{_x['n']} → {_y['k']}/{_y['n']} · 문턱 {b8.get('th:' + l)} → {d8.get('th:' + l)}")
        if LABEL_UNIT == "ho":   # 🆕 v8.1 — 호 판을 고르는 조건(머리말 · 돌리기 전에 정했다): PR-AUC +0.015 이상 또는 음성 오판정 3줄 이상 감소
            _gain = m["dev_macro_prauc_mean"] - b_au; _fewer = b.get("k_neg_flag", 0) - d.get("k_neg_flag", 0)
            _pick = ok and (_gain >= 0.015 or _fewer >= 3)
            print(f"  고르는 조건 — PR-AUC {_gain:+.3f} (≥ +0.015) · 음성 오판정 {-_fewer:+d}줄 (≤ −3) → {'호 판' if _pick else '유형 판 유지'}")

# 🆕 v9 — R0 재현: 같은 학습이므로 6종 dev macro PR-AUC 가 R0 과 같아야 한다 (문턱은 규칙이 달라 견주지 않는다)
if THRESHOLD_MODE == "robust" and LABEL_UNIT == "type":
    _r0 = EXPERIMENT_LOG.get(BASE_KEY)
    print("\n=== 🆕 v9 R0 재현 (같은 학습 · 다시 학습한 것) ===")
    if _r0 is None or not same_dev(_r0):
        print("  로그에 R0(v8.3 기준 실행 · 같은 dev 지문)이 없다 — 머리말의 R0 값(0.794 ± 0.006 · 시드 43)과 눈으로 견준다")
    else:
        _gap = abs(_r0["dev_macro_prauc_mean"] - float(np.mean(devs)))
        print(f"  R0   PR-AUC {_r0['dev_macro_prauc_mean']:.3f} ± {_r0['dev_macro_prauc_std']:.3f} · 시드 {_r0.get('best_seed')} · 재현율 {_r0['dev_cond'].get('k_pos_hit')}/{_r0['dev_cond'].get('n_pos')}"
              f" · 음성 오판정 {_r0['dev_cond'].get('k_neg_flag')}/{_r0['dev_cond'].get('n_neg')} (best_seed 문턱)")
        print(f"  v9   PR-AUC {np.mean(devs):.3f} ± {np.std(devs):.3f} · 시드 {BEST_SEED} · best_seed 문턱이면 재현율 {DEV_RULES['best_seed']['k_pos_hit']}/{dev_cond['n_pos']}"
              f" · 음성 오판정 {DEV_RULES['best_seed']['k_neg_flag']}/{dev_cond['n_neg']}")
        print("    → " + ("✅ 같은 학습이 재현됐다" if _gap < 0.003 else ("🟡 조금 다르다 — GPU 비결정성 범위인지 본다" if _gap < 0.01 else "🔴 다르다 — 스위치 · 올린 파일을 확인한다")))
    print("  위 「사전 고정 기준」은 v9(robust 문턱) ↔ R0(best_seed 문턱)이다 — 문턱 규칙을 바꾼 효과로 읽는다. PR-AUC 채택 조건은 해당 없다(모델이 같다)")

print("\n=== 유형별 dev PR-AUC (시드 평균) ===")
keys = list(CANDS) or list(EXPERIMENT_LOG)
for l in CONFIRMED:
    cells_ = []
    for k in keys:
        dv = [v.get(f"prauc_{l}") for v in EXPERIMENT_LOG[k]["dev_by_seed"].values()]
        dv = [v for v in dv if v is not None]
        cells_.append(f"{np.mean(dv):.3f}" if dv else "N/A")
    print(f"{l:20s} " + "  ".join(f"{c:>7s}" for c in cells_))
print("(열 순서:", [cand_name(k) for k in keys], ")")

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    # v7.2 — 덮어쓰기 방지: 디스크의 기존 로그를 다시 읽어 합친 뒤 저장한다
    if os.path.exists(LOG_PATH):
        with open(LOG_PATH, encoding="utf-8") as f:
            on_disk = json.load(f)
        on_disk.update(EXPERIMENT_LOG)
        EXPERIMENT_LOG = on_disk
    with open(LOG_PATH, "w", encoding="utf-8") as f:
        json.dump(EXPERIMENT_LOG, f, ensure_ascii=False, indent=2, default=float)
    print(f"로그 {len(EXPERIMENT_LOG)}건")
    print("\nDrive 저장:", LOG_PATH)
except Exception as e:
    print("Drive 저장 건너뜀:", e)

## 10. (선택) 시드 폴더 정리

다음 후보를 돌리기 전에 디스크가 부족하면 이 셀로 이번 후보의 시드 폴더를 지운다. 저장(섹션 11)을 할 거면 먼저 저장하고 지운다.

In [ ]:
CLEAN_RUN_DIR = False
if CLEAN_RUN_DIR:
    shutil.rmtree(RUN_DIR, ignore_errors=True)
    print("삭제:", RUN_DIR)
else:
    print("유지:", RUN_DIR, "(CLEAN_RUN_DIR=True로 바꾸면 삭제)")

## 11. 모델 저장 (앱 연결용) — `SAVE_THIS_RUN = True`일 때만

- 🔄 v8.2 — **9칸 그대로 저장한다**(분류층을 자르지 않는다). 순서: 기존 6 → `부당_비교광고` → `비방광고` → `기능성화장품_오인` (팀장 10-06).
- `label_scheme.json` 의 `label_list` 가 9개 · `label_thresholds` 도 9개다. 앱 로더(`app/encoder.py`)는 그대로 읽는다 — 코드 수정이 없다.
  - 확정 8종의 문턱은 dev 에서 고른 값이다. 앱 로더는 `확률 >= threshold` 로 후보를 낸다.
  - **편입 대기**(`기능성화장품_오인`)의 문턱은 `1.01` — 확률이 1을 넘을 수 없으므로 이 유형은 후보로 나가지 않는다.
- 폴더 이름은 앱 로더에 맞춘 `copylane-encoder-kcbert-final`. `config.json` 의 `id2label` 과 `label_list` 가 같은 순서다.
- `label_scheme.json` 에 조건별 학습 행 수 · 뺀 행 수 · 사전 sha · **dev 8종 수치(주입 제외 포함)** · 선택 기준(6종)을 남긴다.
- 저장 zip 은 `<RUN_NAME>.zip` 하나다(Colab · Drive 같은 이름 · `copylane-encoder-kcbert-v9-…-문턱robust.zip`). 🔴 압축은 **저장소 밖**이나 `models/` 아래에 푼다 — `docs/psj/e2e_prototype` 안에 두지 않는다.
- B 회차(`FINAL_TEST = True`)를 돌리면 같은 이름으로 다시 저장된다 — 가중치는 같고 `test_report` 가 채워진다.


In [ ]:
if not SAVE_THIS_RUN:
    print("SAVE_THIS_RUN=False → 저장 건너뜀")
elif LABEL_UNIT == "ho":
    print("호 판 — 앱 로더 형식(유형 9칸)으로 저장하지 않는다. 접기(호 → 유형)를 추론 때 해야 해서다.")
    print("이 판을 고르게 되면 저장 형식(호 로짓 + 접기 표를 함께 · 앱 쪽에서 접기)을 팀장과 정한 뒤 다시 돌려 저장한다.")
else:
    SAVE_DIR = "./copylane-encoder-kcbert-final"
    shutil.rmtree(SAVE_DIR, ignore_errors=True)
    # 🔄 v8.2 (D-321) — 9칸 그대로 저장한다(자르지 않는다). 순서: 기존 6 → 부당 비교 → 비방 → 기능성화장품
    final = best_model.cpu().float()
    _id2l = [final.config.id2label[i] for i in range(final.config.num_labels)]
    assert _id2l == LABEL_LIST and len(LABEL_LIST) == N8 + 1, f"저장 칸이 9칸(확정 8 + 편입 대기)이 아니다: {_id2l}"
    TH_SAVE = {l: float(TH[l]) for l in CONF8}
    TH_SAVE[WAITING[0]] = TH_OFF
    assert list(TH_SAVE) == LABEL_LIST and TH_SAVE[WAITING[0]] > 1 and all(0 < TH_SAVE[l] < 1 for l in CONF8)
    final.save_pretrained(SAVE_DIR, safe_serialization=True)
    tokenizer.save_pretrained(SAVE_DIR)
    assert os.path.exists(f"{SAVE_DIR}/tokenizer.json"), "tokenizer.json이 없다 (앱 로더 필수 파일)"
    with open(f"{SAVE_DIR}/label_scheme.json", "w", encoding="utf-8") as f:
        json.dump({
            "label_list": LABEL_LIST,
            "label_thresholds": TH_SAVE,
            "confirmed_labels": CONF8, "waiting_labels": WAITING, "legacy_confirmed_labels": CONFIRMED,
            "waiting_note": f"편입 대기(D-321) — 칸은 두되 문턱 {TH_OFF} 로 꺼 둔다(확률 >= 문턱이 참이 될 수 없다). 참고 수치는 {WAIT_REF_TH} 기준",
            "selection_note": "회차(조기 종료) · 시드는 기존 6종 dev macro PR-AUC 로 고른다 — 8종으로 넓히지 않는다 (팀장 10-06 · 2차와 수치를 잇는다)",
            "backbone": MODEL_NAME, "notebook": "v9 (3차 골든 · D-296 개정 2 · D-321 확정 8종 · dev 실제 문구 · 문턱 " + THRESHOLD_MODE + ")", "label_unit": LABEL_UNIT,
            "dev_definition": ("합성 포함" if DEV_SCORE_INJECTED else "실제 문구만") + " · " + ("8종 층화" if DEV_STRATIFY_NEW else "6종 층화"),
            "thresholds_best_seed": seed_results[BEST_SEED]["thresholds"], "thresholds_robust": TH_ROBUST,
            "test_scoring_note": "이 골든 판 test 의 5번째 채점(탐색에서 R0 을 쟀다면 6번째) · dev 정의 · 문턱 규칙은 후보 A 의 test 결과를 본 뒤 바꿨다 · 문턱 값은 dev 에서만 나왔다",
            "dev_sha": DEV_SHA, "dev_rows": len(dev_rows), "inj_weight": INJ_WEIGHT, "max_epochs": MAX_EPOCHS, "use_injected": USE_INJECTED,
            "experiment": exp_key, "seed": BEST_SEED,
            "best_epoch": seed_results[BEST_SEED]["best_epoch"],
            "golden_sha256": GOLDEN_SHA, "golden_rows": list(EXPECTED_ROWS), "banned_terms_sha256": BANNED_SHA,
            "train_rows": len(train_rows), "train_by_cond": TRAIN_BY_COND, "train_role_counts": dict(ROLE_COUNTS),
            "train_rule": "조건 M 제외 · D · L 음성 · C · A · B · 조건 없음은 유형이 양성 (D-296 개정 2)"
                          + (" · 유형 없는 위반 제외" if DROP_UNTYPED_POS else " · 유형 없는 위반은 음성") + ("" if USE_WORD_ROWS else " · 낱말 행 제외"),
            "use_word_rows": USE_WORD_ROWS, "drop_untyped_pos": DROP_UNTYPED_POS, "train_injected": TRAIN_INJ,
            "threshold_mode": THRESHOLD_MODE,
            "thresholds_by_seed": {str(s_): seed_results[s_]["thresholds"] for s_ in SEEDS},
            "threshold_note": ("시드 3개 dev 예측의 평균 F1 최대" if THRESHOLD_MODE == "robust" else "가장 좋은 시드의 dev F1 최대") + " · 격자 0.1~0.9(0.025) · dev 양성 10건 미만이면 0.5. 적용은 확률 >= threshold.",
            "scope_note": "확정 8종(D-321 = D-232 의 6종 + 부당_비교광고 · 비방광고) · 유형 단위 출력(D-131). 기능성화장품_오인은 편입 대기(문턱 꺼 둠). "
                          "판정 대상 아님(D-275) · low_conf(M) · 화장품법 13①4 등 유형 없는 위반은 이 인코더가 내지 않는다.",
            "test_report": ({"micro_f1": micro_dev, **bucket_summary, "test8": test8} if FINAL_TEST else None),
            "final_test": FINAL_TEST, "dev_cond": dev_cond, "dev8": dev8,
            "extra_train": EXTRA_TRAIN_FILES, "extra_sha256": {f: EXTRA_SHA[f] for f in EXTRA_TRAIN_FILES}, "extra_rows": len(extra_rows),
        }, f, ensure_ascii=False, indent=2, default=float)
    print("저장:", sorted(os.listdir(SAVE_DIR)), f"· {final.config.num_labels}칸 = 확정 {N8} + 편입 대기 {len(WAITING)}")
    print("label_thresholds:", TH_SAVE)
    zip_path = shutil.make_archive(RUN_NAME, "zip", SAVE_DIR)
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    dst = f"/content/drive/MyDrive/{RUN_NAME}.zip"
    shutil.copy(zip_path, dst)
    print("드라이브 저장:", dst)
    if FINAL_TEST:   # 🆕 v9 — 행별 확률(문구 없음)도 Drive 에 둔다. 보고서의 추가 표는 이 파일로 만든다
        for _f in (CSV_TEST, CSV_DEV):
            shutil.copy(_f, f"/content/drive/MyDrive/{_f}")
        print("드라이브 저장:", CSV_TEST, "·", CSV_DEV)